# PRISM on LUMOS: reference pipeline

This notebook reproduces every experiment of the paper *PRISM: opportunistic osteoporosis screening from lumbar radiographs* on the public LUMOS dataset. Code comments and section titles are in French; this cell gives the English map.

**Before running**: download `lumos_clinical_data.xlsx` and `lumos_npy.zip` from [Zenodo](https://doi.org/10.5281/zenodo.18173664), unzip into `data/LUMOS/`, then run `python scripts/prepare_data.py --lumos-dir data/LUMOS` (see the README). Set `LUMOS_ROOT` to use another location. Run All resumes automatically: preprocessing caches and finished experiments are reloaded from `cache/` and `results_prism/`.

| Notebook section | Paper |
|---|---|
| 1. Configuration | Table II (training and implementation settings) |
| 2. Data and leakage guard | Section III-B (Dataset): DXA-derived variables are never inputs |
| 3, 3.1 Preprocessing, inverted radiographs | Section III-C (Data preprocessing) |
| 4. Metrics, clinical and OST baselines | Section III-I; baselines of Table III |
| 5, 5.1 Encoders, AP/LAT view identification | Section III-D (Image encoders and token extraction) |
| 5.2 Linear probe baseline | Baseline of Table III |
| 6. Spine crop | Section III-C |
| 7. Spinal implants (keep / flag / exclude_train / mask) | Section III-C |
| 8. Token PCA (no labels) | Section III-D |
| 9, 10. PRISM model, losses, training, cross-validation | Sections III-E to III-G, Eq. (1)-(4) |
| 11. Design grid | Experiment 1 |
| 12. Fine-tuning and nested selection | Experiment 1, nested selection |
| 13. Component ablation | Experiment 3 |
| 14, 16. Figures and tables exported for the article | Figures 2-6, Tables |
| 17. Encoder benchmark | Experiment 2 |
| 18. Clinical encoding ablation | Experiment 4 |
| 19. CT encoder comparison | Experiment 5 |

Approximate run time on an Apple Silicon Mac (MPS): RAD-DINO feature extraction 15-30 min; 3-8 min per frozen configuration (3 seeds x 5 folds); several hours per fine-tuned configuration.


# LUMOS-PRISM : classification multimodale de l'ostéoporose (Normal / Ostéopénie / Ostéoporose)

**PRISM** = *Privileged-Regularized, Imaging-attention, Self-distilled Multimodal* model.

Ce notebook entraîne et évalue un modèle à 3 classes à partir des **radiographies lombaires (AP/LAT)**, des **scanners (CT)** quand ils existent et des **données cliniques** (âge, sexe, taille, poids, IMC).

### Les 6 idées clés de l'approche
1. **Pas de fuite de données** : aucune mesure DXA (BMD, BMC, T-score, Z-score, surfaces, dimensions) n'entre dans le modèle.
2. **Information privilégiée (LUPI)** : ces mêmes mesures DXA (≈ 44 valeurs) servent de **cibles auxiliaires** pendant l'entraînement seulement. Le réseau apprend à « voir » la densité osseuse, puis il n'en a plus besoin pour prédire.
3. **Attention spatiale sur la radiographie** : les cartes de caractéristiques d'un CNN pré-entraîné sont agrégées par une attention (*gated attention MIL*). Le modèle apprend seul où regarder (les corps vertébraux), et les cartes d'attention sont interprétables.
4. **Fusion multi-vues + multimodale robuste aux absences** : 1 ou 2 vues, CT présent ou absent (68 % des patients n'en ont pas). Masques de modalité et *modality dropout*.
5. **Auto-distillation CT → radiographie** : pour les patients qui ont un CT, la prédiction « sans CT » doit imiter la prédiction « avec CT ». Le savoir du CT est transféré à la branche radiographie, utilisable chez tous les patients.
6. **Sortie ordinale (CORN) + régression du T-score**, avec un CT décrit aussi par une **densitométrie opportuniste sans segmentation** (HU trabéculaires, corrélation de Spearman ≈ 0,68 avec le T-score).

Le protocole d'évaluation est une **validation croisée à 5 plis par patient** (colonne `fold`), avec une étude d'ablation complète.

### Version 3 : 4 pistes d'amélioration testables en grille + estimation sans biais de sélection
| Piste | Options (`GRID`) |
|---|---|
| **A. Extracteur d'images** | `resnet50` (ImageNet) · `raddino` (RAD-DINO, pré-entraîné sur des radiographies, Microsoft) |
| **B. Région analysée** | `full` (image entière) · `spine` (recadrage automatique sur le rachis lombaire, haute résolution) · `full+spine` (les deux, multi-échelle) |
| **C. Matériel chirurgical** (vis, tiges, ciment) | `keep` (rien) · `flag` (variable « matériel » en entrée) · `exclude_train` (exclu de l'entraînement) · `mask` (zones métalliques masquées dans l'attention) |
| **D. Fine-tuning** | `False` (extracteur gelé) · `True` (ré-entraînement de bout en bout) |

Chaque combinaison est évaluée avec le même protocole, avec **intervalles de confiance à 95 % (bootstrap)**, **test de DeLong** contre la référence, et des résultats séparés pour les patients **sans matériel**.

**Nouveau (v3)** : une **sélection imbriquée** de la configuration (section 12.2). Dans chaque pli, la configuration est choisie uniquement sur la validation interne de ce pli, jamais sur son pli de test. Le score obtenu est une estimation **sans biais de sélection** du pipeline complet « on essaie tout et on garde le meilleur » : c'est ce chiffre qu'il faut rapporter dans un article.

**Sur Mac (Apple Silicon)** : le notebook utilise automatiquement le GPU intégré (MPS). Lancez simplement *Run All*. Une exécution interrompue reprend là où elle s'était arrêtée.

## 0. Installation (une seule fois)
Décommentez la ligne si un paquet manque.

In [ ]:
# %pip install torch torchvision timm transformers scikit-learn scipy pillow pandas openpyxl matplotlib seaborn tqdm

## 1. Configuration

In [ ]:
import os, sys, json, math, time, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# --- Chemins -------------------------------------------------------------------
ROOT   = Path(os.environ.get('LUMOS_ROOT', '../data/LUMOS'))   # dossier LUMOS (voir README : scripts/prepare_data.py)

# Kaggle / Colab : le jeu de données « lumos_kaggle » (≈ 300 Mo, caches seulement) est copié dans un dossier modifiable
KAGGLE = Path('/kaggle/input').exists()
if KAGGLE and 'LUMOS_ROOT' not in os.environ:
    import shutil
    src = next(Path('/kaggle/input').rglob('lumos_master_dataset.csv')).parent
    ROOT = Path('/kaggle/working/LUMOS')
    if not ROOT.exists():
        shutil.copytree(src, ROOT)
        for prev in Path('/kaggle/input').rglob('results_prism'):      # résultats d'une session précédente (reprise)
            if prev.is_dir() and prev.resolve() != (ROOT / 'results_prism').resolve():
                shutil.copytree(prev, ROOT / 'results_prism', dirs_exist_ok=True)
NPY_X  = ROOT / 'lumos_npy' / 'lumos_npy' / 'lumos_x'
NPY_CT = ROOT / 'lumos_npy' / 'lumos_npy' / 'lumos_ct'
CACHE  = ROOT / 'cache'
OUT    = ROOT / 'results_prism'
for d in (CACHE, OUT):
    d.mkdir(parents=True, exist_ok=True)

# --- Prétraitement ---------------------------------------------------------------
XRAY_H, XRAY_W = 512, 320          # radiographie redimensionnée (rapport d'aspect conservé + bandes noires)
CT_SLICES, CT_CROP, CT_OUT = 16, 192, 128   # 16 coupes axiales, recadrage 192 px centré sur le rachis -> 128 px

# --- Extracteurs d'images -------------------------------------------------------------
LOCAL_WEIGHTS = os.environ.get('LUMOS_LOCAL_WEIGHTS')   # optionnel : fichier .pth local pour resnet50
TOKEN_GRID = (16, 10)              # grille de tokens commune à tous les extracteurs

# --- Grille d'expériences (modifiez librement) ------------------------------------------------
GRID = dict(
    backbone = ['resnet50', 'raddino'],
    roi      = ['full', 'spine', 'full+spine'],
    hardware = ['keep', 'flag', 'exclude_train', 'mask'],
)
REFERENCE   = ('resnet50', 'full', 'keep')   # expérience de référence (tests de DeLong)
GRID_SEEDS  = [0, 1, 2]
FINETUNE_GRID  = 'best'            # 'none' | 'best' (fine-tune les FINETUNE_TOP meilleures config. gelées) | 'all'
FINETUNE_TOP   = 2                 # les 2 meilleures configurations gelées sont fine-tunées
FINETUNE_SEEDS = [0]
RUN_NESTED     = True              # estimation imbriquée sans biais de sélection (section 12.2)
RUN_ABLATIONS  = True              # ablation des modalités sur la meilleure configuration gelée
HW_THR = 0.003                     # seuil de détection du matériel (score métal max par patient)
FINAL_MODEL = ('raddino', 'full', 'keep')   # modèle final de l'article (ablations, figures) ; None = meilleur QWK de la grille
CT_BACKBONE = 'raddino'            # encodeur des coupes CT : 'raddino' (même encodeur que les radiographies) | 'resnet50' | 'dinov2'
REUSE_FT_RESNET_CT = True          # fine-tuning : réutiliser les résultats déjà calculés (CT encodé par ResNet-50) au lieu de tout relancer
LANG = 'en'                        # langue des figures et tableaux exportés pour l'article : 'en' ou 'fr'

# --- Entraînement ---------------------------------------------------------------------
N_EPOCHS, PATIENCE = 80, 15
BATCH, LR, WD  = 32, 3e-4, 1e-2
D_MODEL        = 256
LAMBDA_T, LAMBDA_PRIV, LAMBDA_DISTILL = 0.5, 0.5, 0.5
P_DROP_XRAY, P_DROP_CT = 0.15, 0.30   # modality dropout

# Fine-tuning : réglages « rapides » pour un Mac (sur GPU CUDA, vous pouvez augmenter FT_EPOCHS et FT_TRAIN_LAST)
FT_EPOCHS, FT_PATIENCE, FT_BATCH, FT_LR_BB, FT_LR_HEAD = 10, 3, 8, 2e-5, 3e-4
FT_MICRO      = 4      # patients traités à la fois ; les gradients sont accumulés jusqu'à FT_BATCH patients par pas
FT_TRAIN_LAST = 2      # blocs ré-entraînés en fin d'extracteur (RAD-DINO : 2 blocs ViT sur 12 ; ResNet-50 : dernier étage)
FT_GRAD_CKPT  = False  # True = beaucoup moins de mémoire mais ≈ 30 % plus lent (à activer si « MPS out of memory »)
FT_VAL_EVERY  = 1      # validation toutes les N époques

SMOKE = bool(int(os.environ.get('LUMOS_SMOKE', '0')))   # test rapide de bout en bout
if os.environ.get('LUMOS_GRID'):                      # surcharge possible (tests)
    GRID.update(json.loads(os.environ['LUMOS_GRID']))
if os.environ.get('LUMOS_SEEDS'):
    GRID_SEEDS = json.loads(os.environ['LUMOS_SEEDS'])
if os.environ.get('LUMOS_FT'):
    FINETUNE_GRID = os.environ['LUMOS_FT']
if os.environ.get('LUMOS_FINAL'):
    FINAL_MODEL = tuple(json.loads(os.environ['LUMOS_FINAL']))
if os.environ.get('LUMOS_CT'):
    CT_BACKBONE = os.environ['LUMOS_CT']
if os.environ.get('LUMOS_FT_TOP'):
    FINETUNE_TOP = int(os.environ['LUMOS_FT_TOP'])
if SMOKE:
    GRID_SEEDS, N_EPOCHS, PATIENCE = [0], 2, 1
    FT_EPOCHS, FT_PATIENCE, FT_BATCH, FT_MICRO = 1, 1, 2, 1

os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')   # Mac : repli CPU pour les rares opérations non supportées par MPS
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_everything(0)
print('ROOT   :', ROOT, '| existe :', ROOT.exists())
print('DEVICE :', DEVICE, '| torch', torch.__version__, '| SMOKE =', SMOKE)
if DEVICE == 'cpu':
    print('⚠️ Aucun GPU détecté : tout tournera sur CPU (beaucoup plus lent). Sur Mac, vérifiez que PyTorch ≥ 2.0 est installé pour Apple Silicon.')

def free_gpu_memory():
    import gc
    gc.collect()
    if DEVICE == 'mps':
        torch.mps.empty_cache()
    elif DEVICE == 'cuda':
        torch.cuda.empty_cache()

## 2. Données et garde-fou contre la fuite de données

In [ ]:
df = pd.read_csv(ROOT / 'lumos_master_dataset.csv')
df['sex_f'] = (df['gender'] == 'Female').astype(float)

def _stems(s):
    if not isinstance(s, str) or not s.strip():
        return []
    return [Path(p).stem for p in s.split(';') if p.strip()]
df['xray_stems'] = df['xray_paths'].apply(_stems)
df['ct_stem'] = df['ct_paths'].apply(lambda s: (_stems(s) or [None])[0])

CLIN = ['age', 'sex_f', 'height', 'weight', 'BMI']                       # entrées autorisées
PRIV = ['bmd', 't_value'] + [c for c in df.columns
        if c.startswith(('bmd_', 'bmc_', 's_L', 'T_L', 'Z_L', 'avgh_', 'avgw_'))]   # cibles auxiliaires uniquement
CLASSES = ['Normal', 'Ostéopénie', 'Ostéoporose']

assert not set(CLIN) & set(PRIV), 'Fuite : une variable DXA est utilisée en entrée !'
assert df['patient_id'].is_unique
# la cible est bien définie par le T-score (seuils OMS)
chk = np.select([df.t_value >= -1, df.t_value > -2.5], [0, 1], 2)
print('Cible cohérente avec t_value :', (chk == df.Osteoporosis).mean() * 100, '%')
print(f'{len(df)} patients | radiographies : {df.xray_stems.str.len().sum()} | CT : {df.ct_stem.notna().sum()}')
print(f'{len(PRIV)} cibles privilégiées (jamais en entrée)')
display(pd.crosstab(df['fold'], df['Osteoporosis'].map(dict(enumerate(CLASSES)))))

## 3. Prétraitement et cache
Les fichiers `.npy` d'origine sont lourds (29 Go). On crée une seule fois un cache compact :
- **radiographies** : normalisation par percentiles (0,5–99,5 %), redimensionnement 512×320 avec bandes noires → PNG ;
- **CT** : 16 coupes axiales recadrées automatiquement sur le rachis (fenêtre osseuse −200 à 1000 HU) ;
- **CT** : descripteurs de **densitométrie opportuniste** (HU trabéculaires du corps vertébral), sans segmentation.

Les étapes déjà faites sont sautées.

In [ ]:
from PIL import Image
from scipy import ndimage as ndi

X_CACHE  = CACHE / f'xray_{XRAY_H}x{XRAY_W}'
CT_CACHE = CACHE / f'ct_crops_{CT_SLICES}x{CT_OUT}'
HU_CACHE = CACHE / 'ct_hu'
for d in (X_CACHE, CT_CACHE, HU_CACHE):
    d.mkdir(parents=True, exist_ok=True)

def xray_preprocess(a, H=XRAY_H, W=XRAY_W):
    a = a.astype(np.float32)
    lo, hi = np.percentile(a[::4, ::4], [0.5, 99.5])
    a = np.clip((a - lo) / (hi - lo + 1e-6), 0, 1)
    h, w = a.shape
    s = min(H / h, W / w)
    nh, nw = max(1, round(h * s)), max(1, round(w * s))
    im = np.asarray(Image.fromarray((a * 255).astype(np.uint8)).resize((nw, nh), Image.BILINEAR))
    out = np.zeros((H, W), np.uint8)
    y0, x0 = (H - nh) // 2, (W - nw) // 2
    out[y0:y0 + nh, x0:x0 + nw] = im
    return out

def spine_center(sl):
    """Centre du rachis sur une coupe axiale (HU) : médiane de l'os dense près de la ligne médiane du corps."""
    H, W = sl.shape
    body = sl > -300
    lab, n = ndi.label(body)
    if n == 0:
        return None
    sizes = ndi.sum(body, lab, range(1, n + 1))
    body = lab == (1 + int(np.argmax(sizes)))
    cx = np.nonzero(body)[1].mean()
    dense = (ndi.uniform_filter(sl, 3) > 300) & body
    dense[:, :int(max(cx - 0.12 * W, 0))] = False
    dense[:, int(min(cx + 0.12 * W, W)):] = False
    if dense.sum() < 50:
        return None
    ys, xs = np.nonzero(dense)
    return float(np.median(ys)), float(np.median(xs))

def ct_slice_crops(vol, n_slices=CT_SLICES, crop=CT_CROP, out=CT_OUT):
    Z = vol.shape[0]
    zs = np.linspace(0.1 * (Z - 1), 0.9 * (Z - 1), n_slices).round().astype(int)
    crops, centers = [], []
    for z in zs:
        sl = vol[z].astype(np.float32)
        c = spine_center(sl)
        if c is None:
            c = centers[-1] if centers else (sl.shape[0] * 0.6, sl.shape[1] * 0.5)
        centers.append(c)
        cy, cx = int(c[0]) - crop // 6, int(c[1])          # léger décalage antérieur -> corps vertébral
        pad = np.pad(sl, crop, constant_values=-1024)
        win = pad[cy + crop - crop // 2: cy + crop + crop // 2, cx + crop - crop // 2: cx + crop + crop // 2]
        win = np.clip((win + 200) / 1200, 0, 1)
        crops.append(np.asarray(Image.fromarray((win * 255).astype(np.uint8)).resize((out, out), Image.BILINEAR)))
    return np.stack(crops), np.array(centers)

def ct_hu_features(vol, n_slices=24, half=40):
    """Densitométrie opportuniste sans segmentation : statistiques HU de l'os dans une boîte
    placée en avant des éléments postérieurs (donc dans le corps vertébral)."""
    Z = vol.shape[0]
    zs = np.linspace(0.1 * (Z - 1), 0.9 * (Z - 1), n_slices).round().astype(int)
    med, p25, frac = [], [], []
    for z in zs:
        sl = vol[z].astype(np.float32)
        c = spine_center(sl)
        if c is None:
            continue
        cy, cx = int(c[0]), int(c[1])
        box = sl[max(cy - 3 * half, 0): max(cy - half // 2, 1), max(cx - half, 0): cx + half]
        if box.size == 0:
            continue
        sm = ndi.gaussian_filter(box, 1.0)
        bone = sm[(sm > 60) & (sm < 1200)]
        if bone.size < 100:
            continue
        med.append(np.median(bone)); p25.append(np.percentile(bone, 25)); frac.append((sm > 200).mean())
    if len(med) < 3:
        return dict(ct_hu_med=np.nan, ct_hu_p25=np.nan, ct_bone_frac=np.nan)
    return dict(ct_hu_med=float(np.median(med)), ct_hu_p25=float(np.median(p25)), ct_bone_frac=float(np.median(frac)))

# --- construction du cache (sauté si déjà fait) ---------------------------------------
x_stems = sorted({s for L in df.xray_stems for s in L})
todo = [s for s in x_stems if not (X_CACHE / f'{s}.png').exists()]
for s in tqdm(todo, desc='Cache radiographies'):
    Image.fromarray(xray_preprocess(np.load(NPY_X / f'{s}.npy'))).save(X_CACHE / f'{s}.png')

ct_stems = sorted(df.ct_stem.dropna().unique())
todo = [s for s in ct_stems if not ((CT_CACHE / f'{s}.npy').exists() and (HU_CACHE / f'{s}.json').exists())]
for s in tqdm(todo, desc='Cache CT'):
    v = np.load(NPY_CT / f'{s}.npy')
    np.save(CT_CACHE / f'{s}.npy', ct_slice_crops(v)[0])
    f = ct_hu_features(v); f['n_slices_raw'] = int(v.shape[0])
    json.dump(f, open(HU_CACHE / f'{s}.json', 'w'))

HU_COLS = ['ct_hu_med', 'ct_hu_p25', 'ct_bone_frac']
hu = pd.DataFrame([dict(ct_stem=s, **json.load(open(HU_CACHE / f'{s}.json'))) for s in ct_stems])
df = df.drop(columns=[c for c in HU_COLS if c in df], errors='ignore').merge(hu[['ct_stem'] + HU_COLS], on='ct_stem', how='left')
print('Radiographies en cache :', len(list(X_CACHE.glob('*.png'))), '| CT en cache :', len(list(CT_CACHE.glob('*.npy'))))
print('Corrélation de Spearman HU (CT) ↔ T-score :')
print(df[HU_COLS + ['t_value']].corr(method='spearman')['t_value'].round(3).drop('t_value').to_string())

### 3.1 Détection automatique des radiographies inversées
Environ 20 patients ont des images en négatif (os sombre, métal noir). Deux indices sont combinés :
1. un **contraste centre/bords** (le rachis central est clair sur une image normale) ;
2. un **détecteur auto-supervisé** : une régression logistique apprend à distinguer chaque image de sa version inversée artificiellement, puis on l'applique aux vraies images.

La décision est propagée aux deux vues d'un même patient si l'indice est fort. Vous pouvez corriger à la main avec `MANUAL_INVERT` / `MANUAL_KEEP` après avoir regardé la mosaïque.

In [ ]:
from sklearn.linear_model import LogisticRegression

MANUAL_INVERT, MANUAL_KEEP = set(), set()     # ex. {'lumos_x_084_1'}

def _content_bbox(a):
    rows, cols = np.where(a.max(1) > 0)[0], np.where(a.max(0) > 0)[0]
    return rows[0], rows[-1] + 1, cols[0], cols[-1] + 1

def center_edge_score(a):
    y0, y1, x0, x1 = _content_bbox(a)
    a = a[y0:y1, x0:x1].astype(np.float32) / 255
    h, w = a.shape
    e = max(1, int(.06 * w))
    return a[int(.2 * h):int(.8 * h), int(.35 * w):int(.65 * w)].mean() - np.r_[a[:, :e].ravel(), a[:, -e:].ravel()].mean()

inv_file = CACHE / 'xray_inversion.csv'
if inv_file.exists():
    inv = pd.read_csv(inv_file)
else:
    ims  = [np.asarray(Image.open(X_CACHE / f'{s}.png')) for s in tqdm(x_stems, desc='Analyse polarité')]
    S    = np.array([center_edge_score(a) for a in ims])
    small = np.stack([np.asarray(Image.fromarray(a).resize((32, 48), Image.BILINEAR)) for a in ims]).astype(np.float32) / 255
    mask  = np.stack([np.asarray(Image.fromarray(a).resize((32, 48), Image.NEAREST)) > 0 for a in ims])
    small_inv = np.where(mask, 1 - small, 0)
    def _feat(A):
        return np.c_[A.reshape(len(A), -1), np.stack([np.histogram(a[a > 0], bins=16, range=(0, 1), density=True)[0] for a in A])]
    clf = LogisticRegression(C=0.1, max_iter=3000).fit(np.r_[_feat(small), _feat(small_inv)],
                                                        np.r_[np.zeros(len(small)), np.ones(len(small))])
    P = clf.predict_proba(_feat(small))[:, 1]
    inv = pd.DataFrame(dict(stem=x_stems, score_center_edge=S, p_inverted=P))
    inv['pid'] = inv.stem.str.slice(8, 11)
    strong = inv.loc[(inv.score_center_edge < -0.30) | (inv.p_inverted > 0.85), 'pid']
    inv['inverted'] = (inv.score_center_edge < -0.18) | (inv.p_inverted > 0.60) | inv.pid.isin(strong)
    inv.to_csv(inv_file, index=False)
inv.loc[inv.stem.isin(MANUAL_INVERT), 'inverted'] = True
inv.loc[inv.stem.isin(MANUAL_KEEP), 'inverted'] = False
INVERTED = set(inv.loc[inv.inverted, 'stem'])
print(f'{len(INVERTED)} radiographies détectées comme inversées ({inv[inv.inverted].pid.nunique()} patients)')

def load_xray(stem):
    a = np.asarray(Image.open(X_CACHE / f'{stem}.png')).copy()
    if stem in INVERTED:
        y0, y1, x0, x1 = _content_bbox(a)
        a[y0:y1, x0:x1] = 255 - a[y0:y1, x0:x1]
    return a

def show_mosaic(stems, ncol=12, title=''):
    stems = list(stems)
    if not stems:
        print('(aucune image)'); return
    nrow = int(math.ceil(len(stems) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 1.2, nrow * 2.0))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis('off')
    for ax, s in zip(np.atleast_1d(axes).ravel(), stems):
        ax.imshow(load_xray(s), cmap='gray'); ax.set_title(s[8:], fontsize=7)
    fig.suptitle(title); plt.tight_layout(); plt.show()

show_mosaic(sorted(INVERTED), title='Images inversées (après correction) — vérifiez visuellement')

## 4. Métriques et référence clinique
**Métrique principale : Kappa pondéré quadratique (QWK)**, adaptée à une cible ordinale. On rapporte aussi l'accuracy équilibrée, le F1 macro, l'AUC macro (one-vs-rest), l'AUC « ostéoporose contre le reste » et l'erreur sur le T-score.

In [ ]:
from sklearn.metrics import (cohen_kappa_score, f1_score, balanced_accuracy_score, accuracy_score,
                             roc_auc_score, confusion_matrix, recall_score)
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

def metrics3(y, prob, t_true=None, t_pred=None, pred=None):
    y = np.asarray(y); prob = np.asarray(prob)
    pred = prob.argmax(1) if pred is None else np.asarray(pred)
    m = dict(acc=accuracy_score(y, pred), bal_acc=balanced_accuracy_score(y, pred),
             f1_macro=f1_score(y, pred, average='macro'), qwk=cohen_kappa_score(y, pred, weights='quadratic'),
             auc_macro=roc_auc_score(y, prob, multi_class='ovr', average='macro'),
             auc_osteoporose=roc_auc_score(y == 2, prob[:, 2]),
             sens_osteoporose=recall_score(y == 2, pred == 2), spec_osteoporose=recall_score(y != 2, pred != 2))
    if t_pred is not None:
        m['t_mae'] = float(np.mean(np.abs(np.asarray(t_true) - np.asarray(t_pred))))
        m['t_spearman'] = float(spearmanr(t_true, t_pred).correlation)
    return m

FOLDS = sorted(df['fold'].unique())
y_all = df['Osteoporosis'].values

# Référence : régression logistique multinomiale sur les 5 variables cliniques
oof_clin = np.zeros((len(df), 3))
for k in FOLDS:
    tr, te = df.fold != k, df.fold == k
    sc = StandardScaler().fit(df.loc[tr, CLIN])
    lr = LogisticRegression(max_iter=3000).fit(sc.transform(df.loc[tr, CLIN]), y_all[tr])
    oof_clin[te] = lr.predict_proba(sc.transform(df.loc[te, CLIN]))
RESULTS = {'Référence clinique (LR)': metrics3(y_all, oof_clin)}

# Référence clinique validée : score OST (Osteoporosis Self-assessment Tool) = 0,2 × (poids en kg − âge)
df['OST'] = 0.2 * (df['weight'] - df['age'])
oof_ost = np.zeros((len(df), 3))
for k in FOLDS:
    tr, te = df.fold != k, df.fold == k
    lr = LogisticRegression(max_iter=3000).fit(df.loc[tr, ['OST']], y_all[tr])
    oof_ost[te] = lr.predict_proba(df.loc[te, ['OST']])
RESULTS['Score OST'] = metrics3(y_all, oof_ost)
print(f"AUC ostéoporose du score OST brut (sans apprentissage) : {roc_auc_score(y_all == 2, -df['OST']):.3f}")
pd.DataFrame(RESULTS).T.round(3)

## 5. Extracteurs d'images et cartes de caractéristiques
Deux extracteurs sont disponibles (piste A) :
- **`resnet50`** : ResNet-50 entraîné sur ImageNet (photos ordinaires) ;
- **`raddino`** : **RAD-DINO** (Microsoft), un ViT-B/14 entraîné par auto-supervision sur des centaines de milliers de **radiographies**. Ses tokens de patchs sont ramenés à la même grille 16×10.

On garde la **carte spatiale** (16×10 positions), pas un simple vecteur moyen, ainsi que la version **miroir** (augmentation et TTA). Le CT est encodé par `CT_BACKBONE` (RAD-DINO par défaut), le même pour toutes les expériences : les comparaisons d'extracteurs ne portent donc que sur la radiographie.

In [ ]:
import timm
import torch.nn as nn
import torch.nn.functional as F

BACKBONES = {
    'resnet50':        dict(kind='timm', name='resnet50.a1_in1k',               size=(512, 320), pretrain='ImageNet-1k, supervisé'),
    'densenet121':     dict(kind='timm', name='densenet121.tv_in1k',            size=(512, 320), pretrain='ImageNet-1k, supervisé'),
    'efficientnet_b3': dict(kind='timm', name='efficientnet_b3.ra2_in1k',       size=(512, 320), pretrain='ImageNet-1k, supervisé'),
    'convnext_base':   dict(kind='timm', name='convnext_base.fb_in22k_ft_in1k', size=(512, 320), pretrain='ImageNet-22k, supervisé'),
    'dinov2':          dict(kind='hf',   name='facebook/dinov2-base',           size=(518, 322), pretrain='photos naturelles (LVD-142M), auto-supervisé'),
    'raddino':         dict(kind='hf',   name='microsoft/rad-dino',             size=(518, 322), pretrain='radiographies, auto-supervisé'),
}
RANDOM_WEIGHTS = bool(os.environ.get('LUMOS_RANDOM_WEIGHTS') or os.environ.get('LUMOS_RANDOM_RADDINO'))   # tests hors-ligne uniquement

class Backbone(nn.Module):
    """Extracteur générique : image (B,1,H,W) dans [0,1] -> carte (B,C,16,10)."""
    def __init__(self, key):
        super().__init__()
        self.key, spec = key, BACKBONES[key]
        self.kind, self.size = spec['kind'], spec['size']
        if self.kind == 'timm':
            if LOCAL_WEIGHTS and key == 'resnet50':
                self.net = timm.create_model('resnet50', pretrained=False, num_classes=0, global_pool='')
                sd = torch.load(LOCAL_WEIGHTS, map_location='cpu')
                self.net.load_state_dict({k: v for k, v in sd.items() if not k.startswith('fc.')}, strict=False)
            else:
                self.net = timm.create_model(spec['name'], pretrained=not RANDOM_WEIGHTS, num_classes=0, global_pool='')
            cfg = getattr(self.net, 'pretrained_cfg', {}) or {}
            mean, std = tuple(cfg.get('mean', (0.485, 0.456, 0.406))), tuple(cfg.get('std', (0.229, 0.224, 0.225)))
        else:
            from transformers import AutoModel, AutoImageProcessor, Dinov2Config, Dinov2Model
            if RANDOM_WEIGHTS:                              # test hors-ligne uniquement (poids aléatoires)
                self.net = Dinov2Model(Dinov2Config(hidden_size=768, num_hidden_layers=12, num_attention_heads=12,
                                                    patch_size=14, image_size=518))
                mean, std = (0.5,) * 3, (0.25,) * 3
            else:
                self.net = AutoModel.from_pretrained(spec['name'])
                proc = AutoImageProcessor.from_pretrained(spec['name'])
                mean, std = tuple(proc.image_mean), tuple(proc.image_std)
            self.patch = self.net.config.patch_size
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(std).view(1, 3, 1, 1))

    def prepare_finetune(self, n_last=FT_TRAIN_LAST, grad_ckpt=FT_GRAD_CKPT):
        """Gèle l'extracteur sauf ses derniers blocs, et active le gradient checkpointing (économie de mémoire GPU)."""
        for p in self.net.parameters():
            p.requires_grad = False
        if self.kind == 'timm':
            names = [n for n, _ in self.net.named_parameters()]
            if self.key == 'resnet50':
                stages = ['layer1', 'layer2', 'layer3', 'layer4'][-max(1, min(4, n_last // 2)):]
                keep = lambda n: n.split('.')[0] in stages
            else:                                         # autres CNN : dernier quart des paramètres
                last = set(names[int(len(names) * 0.75):])
                keep = lambda n: n in last
            for n, p in self.net.named_parameters():
                if keep(n):
                    p.requires_grad = True
            if grad_ckpt:
                self.net.set_grad_checkpointing(True)
        else:
            for blk in self.net.encoder.layer[-n_last:]:
                for p in blk.parameters():
                    p.requires_grad = True
            for p in self.net.layernorm.parameters():
                p.requires_grad = True
            if grad_ckpt:
                try:
                    self.net.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
                except TypeError:                     # anciennes versions de transformers
                    self.net.gradient_checkpointing_enable()
        self.grad_ckpt = grad_ckpt
        n_tr = sum(p.numel() for p in self.net.parameters() if p.requires_grad)
        print(f'  fine-tuning {self.key} : {n_tr / 1e6:.1f} M paramètres entraînés sur {sum(p.numel() for p in self.net.parameters()) / 1e6:.1f} M')

    def forward(self, x):
        if tuple(x.shape[-2:]) != tuple(self.size):
            x = F.interpolate(x, size=self.size, mode='bilinear', align_corners=False)
        x = (x.repeat(1, 3, 1, 1) - self.mean) / self.std
        if self.training and getattr(self, 'grad_ckpt', False):
            x = x.detach().requires_grad_()            # nécessaire pour que le checkpointing propage les gradients
        if self.kind == 'timm':
            f = self.net.forward_features(x)              # carte (B, C, h, w) de la dernière couche
        else:
            hs = self.net(pixel_values=x).last_hidden_state
            gh, gw = self.size[0] // self.patch, self.size[1] // self.patch
            f = hs[:, -gh * gw:].transpose(1, 2).reshape(x.shape[0], -1, gh, gw)   # ignore CLS / registres
        try:
            return F.adaptive_avg_pool2d(f, TOKEN_GRID)
        except RuntimeError:                      # MPS : tailles non divisibles -> calcul sur CPU (gradients conservés)
            return F.adaptive_avg_pool2d(f.cpu(), TOKEN_GRID).to(f.device)

def to_tensor01(batch_u8):
    return torch.from_numpy(np.stack(batch_u8)).float().div(255).unsqueeze(1)

def feat_file(bb, roi):
    return CACHE / f'feat_{bb}_{roi}.npy'

def extract_features(bb, roi, loader, stems=None, bs=8):
    """Carte de caractéristiques (N_img, 2 [orig/miroir], C, 16, 10) en float16, en cache disque."""
    f = feat_file(bb, roi)
    if f.exists():
        return
    stems = stems or x_stems
    net = Backbone(bb).eval().to(DEVICE)
    with torch.no_grad():
        C = net(to_tensor01([loader(stems[0])]).to(DEVICE)).shape[1]
    tmp = f.with_suffix('.tmp.npy')
    out = np.lib.format.open_memmap(tmp, mode='w+', dtype=np.float16, shape=(len(stems), 2, C) + TOKEN_GRID)
    with torch.no_grad():
        for i in tqdm(range(0, len(stems), bs), desc=f'Features {bb}/{roi}'):
            x = to_tensor01([loader(s) for s in stems[i:i + bs]]).to(DEVICE)
            out[i:i + bs, 0] = net(x).cpu().numpy().astype(np.float16)
            out[i:i + bs, 1] = net(torch.flip(x, dims=[3])).cpu().numpy().astype(np.float16)
    out.flush(); del out
    os.replace(tmp, f)

# Radiographies entières (ResNet-50) : nécessaires pour identifier les vues
extract_features('resnet50', 'full', load_xray)

# CT : 16 coupes par patient -> un vecteur par coupe, avec l'encodeur CT_BACKBONE (gelé)
CT_VIT_SIZE = 518                                   # ViT : coupe carrée agrandie -> 37 × 37 patchs de 14 px

def ct_feat_file(bb):
    return CACHE / f'feat_ct_{bb}.npy'

def extract_ct_features(bb):
    """Caractéristiques par coupe (N_ct, 16, C) en float16, en cache disque.
    CNN : pooling moyen global de la dernière carte ; ViT : moyenne des tokens de patch."""
    f = ct_feat_file(bb)
    if f.exists():
        return torch.from_numpy(np.load(f))
    net = Backbone(bb).eval().to(DEVICE)
    out = None
    with torch.no_grad():
        for i, s in enumerate(tqdm(ct_stems, desc=f'Features CT {bb}')):
            x = to_tensor01(list(np.load(CT_CACHE / f'{s}.npy'))).to(DEVICE)      # (16, 1, 128, 128)
            if net.kind == 'timm':
                v = net.net((x.repeat(1, 3, 1, 1) - net.mean) / net.std).mean((2, 3))
            else:
                x = F.interpolate(x, size=(CT_VIT_SIZE, CT_VIT_SIZE), mode='bilinear', align_corners=False)
                hs = net.net(pixel_values=(x.repeat(1, 3, 1, 1) - net.mean) / net.std).last_hidden_state
                v = hs[:, -(CT_VIT_SIZE // net.patch) ** 2:].mean(1)
            v = v.float().cpu().numpy()
            if out is None:
                out = np.zeros((len(ct_stems), CT_SLICES, v.shape[1]), np.float16)
            out[i] = v.astype(np.float16)
    np.save(f, out)
    del net; free_gpu_memory()
    return torch.from_numpy(out)

CTF = extract_ct_features(CT_BACKBONE)
CT_ACTIVE = CT_BACKBONE                             # encodeur CT des tokens actuellement chargés (sert au nom des sauvegardes)
X_IDX  = {s: i for i, s in enumerate(x_stems)}
CT_IDX = {s: i for i, s in enumerate(ct_stems)}
print('Features CT :', tuple(CTF.shape))

### 5.1 Identification non supervisée des vues (AP / LAT)
Le nom de fichier (`_0`, `_1`) n'indique pas la vue. Un k-means à 2 groupes sur les caractéristiques sépare les vues. Le groupe le plus **symétrique gauche-droite** est la vue de face (AP). **Contrôle** : chez les patients avec 2 images, les deux images devraient tomber dans des groupes différents.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
XF_full = np.load(feat_file('resnet50', 'full'), mmap_mode='r')
gap = np.concatenate([XF_full[i:i + 64, 0].astype(np.float32).mean((2, 3)) for i in range(0, len(XF_full), 64)])
km = KMeans(2, n_init=10, random_state=0).fit_predict(PCA(32, random_state=0).fit_transform(StandardScaler().fit_transform(gap)))

def lr_symmetry(a):
    y0, y1, x0, x1 = _content_bbox(a)
    b = a[y0:y1:4, x0:x1:4].astype(np.float32)
    return np.corrcoef(b.ravel(), b[:, ::-1].ravel())[0, 1]
sym = np.array([lr_symmetry(load_xray(s)) for s in tqdm(x_stems, desc='Symétrie')])
AP_CLUSTER = int(np.argmax([sym[km == k].mean() for k in (0, 1)]))
VIEW = (km != AP_CLUSTER).astype(int)            # 0 = AP (face), 1 = LAT (profil)
VIEW_OF = dict(zip(x_stems, VIEW))
pairs = [L for L in df.xray_stems if len(L) == 2]
print(f'AP : {(VIEW == 0).sum()} images | LAT : {(VIEW == 1).sum()} images')
print(f'Patients à 2 vues dont les vues diffèrent : {np.mean([VIEW_OF[a] != VIEW_OF[b] for a, b in pairs]) * 100:.1f} %')
for v, nm in ((0, 'AP (face)'), (1, 'LAT (profil)')):
    show_mosaic([s for s in x_stems if VIEW_OF[s] == v][:12], title=f'Vue {nm}')

### 5.2 Référence 2 : sonde linéaire (caractéristiques globales + clinique)
Avant le modèle complet, une référence simple et forte : moyenne des caractéristiques globales des vues, ACP à 64 dimensions, puis régression logistique.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
GAP_OF = dict(zip(x_stems, gap))
P_GAP = np.stack([np.mean([GAP_OF[s] for s in L], 0) if L else np.full(gap.shape[1], np.nan) for L in df.xray_stems])
P_GAP = np.where(np.isnan(P_GAP), np.nanmean(P_GAP, 0), P_GAP)
XPROBE = np.c_[P_GAP, df[CLIN].values]
nG = P_GAP.shape[1]
oof_probe = np.zeros((len(df), 3))
for k in FOLDS:
    tr, te = (df.fold != k).values, (df.fold == k).values
    ct = ColumnTransformer([('img', make_pipeline(StandardScaler(), PCA(64, random_state=0)), list(range(nG))),
                            ('clin', StandardScaler(), list(range(nG, nG + len(CLIN))))])
    clf = make_pipeline(ct, LogisticRegression(C=0.05, max_iter=3000)).fit(XPROBE[tr], y_all[tr])
    oof_probe[te] = clf.predict_proba(XPROBE[te])
RESULTS['Sonde linéaire (image + clinique)'] = metrics3(y_all, oof_probe)
pd.DataFrame(RESULTS).T.round(3)

## 6. Piste B : recadrage automatique sur le rachis lombaire
Pour chaque radiographie, la **trajectoire du rachis** est suivie bande par bande (plateaux vertébraux + os clair, lissée par programmation dynamique). On recadre ensuite une boîte centrée sur le rachis lombaire, approximativement L1–L4 : 62 % de la hauteur de l'image, centrée à 44 % depuis le haut, rapport 1,6.

Le recadrage est fait **dans l'image d'origine en haute résolution** (≈ 2× plus de détails que l'image entière réduite), ce qui est utile pour la texture trabéculaire.

⚠️ C'est une **heuristique** : elle ne détecte pas exactement les niveaux L1–L4 (il faudrait un détecteur de vertèbres entraîné). Vérifiez la mosaïque. Les boîtes peuvent être corrigées à la main dans `ROI_OVERRIDE`.

In [ ]:
def spine_path(a, lateral, n_bands=16, lam=0.6):
    y0, y1, x0, x1 = _content_bbox(a)
    b = ndi.gaussian_filter(a[y0:y1, x0:x1].astype(np.float32) / 255, 1.5)
    h, w = b.shape
    gy, gx = np.abs(ndi.sobel(b, 0)), np.abs(ndi.sobel(b, 1))
    M = gy + 0.7 * b - (0.0 if lateral else 0.5) * gx
    M = ndi.uniform_filter1d(M, max(3, int(.12 * w)), axis=1)
    edges = np.linspace(0, h, n_bands + 1).astype(int)
    B = np.stack([M[edges[i]:edges[i + 1]].mean(0) for i in range(n_bands)])
    lo, hi = int(.12 * w), int(.88 * w)
    B = (B - B[:, lo:hi].mean(1, keepdims=True)) / (B[:, lo:hi].std(1, keepdims=True) + 1e-6)
    B[:, :lo] = -9; B[:, hi:] = -9
    xs = np.arange(w); cost = B[0].copy(); back = np.zeros((n_bands, w), int)
    for i in range(1, n_bands):
        pen = cost[None, :] - lam * np.abs(xs[:, None] - xs[None, :]) / (0.05 * w)
        back[i] = pen.argmax(1); cost = B[i] + pen.max(1)
    p = np.zeros(n_bands, int); p[-1] = cost.argmax()
    for i in range(n_bands - 1, 0, -1):
        p[i - 1] = back[i, p[i]]
    return p + x0, edges + y0

def spine_box(a, lateral, aspect=XRAY_H / XRAY_W, rel_h=0.62, rel_cy=0.44):
    y0, y1, x0, x1 = _content_bbox(a)
    h = y1 - y0
    p, _ = spine_path(a, lateral)
    xc = float(np.median(p[2:13]))
    bh = rel_h * h; bw = bh / aspect; cy = y0 + rel_cy * h
    return cy - bh / 2, cy + bh / 2, xc - bw / 2, xc + bw / 2

ROI_OVERRIDE = {}          # ex. {'lumos_x_123_1': (y0, y1, x0, x1)} en coordonnées de l'image 512×320
roi_file = CACHE / 'roi_boxes.csv'
if roi_file.exists():
    roi_df = pd.read_csv(roi_file)
else:
    rows = []
    for s in tqdm(x_stems, desc='Boîtes rachis'):
        rows.append((s,) + tuple(spine_box(load_xray(s), VIEW_OF[s] == 1)))
    roi_df = pd.DataFrame(rows, columns=['stem', 'by0', 'by1', 'bx0', 'bx1'])
    roi_df.to_csv(roi_file, index=False)
ROI_BOX = {r.stem: (r.by0, r.by1, r.bx0, r.bx1) for r in roi_df.itertuples()}
ROI_BOX.update(ROI_OVERRIDE)

R_CACHE = CACHE / f'xray_roi_{XRAY_H}x{XRAY_W}'
R_CACHE.mkdir(exist_ok=True)

def make_roi_image(stem):
    """Recadre la boîte (définie sur l'image 512×320) dans l'image d'origine pleine résolution."""
    raw = np.load(NPY_X / f'{stem}.npy').astype(np.float32)
    h, w = raw.shape
    s = min(XRAY_H / h, XRAY_W / w)
    oy, ox = (XRAY_H - max(1, round(h * s))) // 2, (XRAY_W - max(1, round(w * s))) // 2
    by0, by1, bx0, bx1 = ROI_BOX[stem]
    Y0, Y1 = int(round((by0 - oy) / s)), int(round((by1 - oy) / s))
    X0, X1 = int(round((bx0 - ox) / s)), int(round((bx1 - ox) / s))
    lo, hi = np.percentile(raw[::4, ::4], [0.5, 99.5])
    a = np.clip((raw - lo) / (hi - lo + 1e-6), 0, 1)
    if stem in INVERTED:
        a = 1 - a
    crop = np.zeros((Y1 - Y0, X1 - X0), np.float32)
    sy0, sy1, sx0, sx1 = max(Y0, 0), min(Y1, h), max(X0, 0), min(X1, w)
    crop[sy0 - Y0:sy1 - Y0, sx0 - X0:sx1 - X0] = a[sy0:sy1, sx0:sx1]
    return np.asarray(Image.fromarray((crop * 255).astype(np.uint8)).resize((XRAY_W, XRAY_H), Image.BILINEAR))

NEED_ROI = any('spine' in r for r in GRID['roi'])
todo = [s for s in x_stems if not (R_CACHE / f'{s}.png').exists()] if NEED_ROI else []
for s in tqdm(todo, desc='Cache recadrages rachis'):
    Image.fromarray(make_roi_image(s)).save(R_CACHE / f'{s}.png')

def load_roi(stem):
    return np.asarray(Image.open(R_CACHE / f'{stem}.png'))

def show_boxes(stems, ncol=12, title=''):
    nrow = int(math.ceil(len(stems) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 1.2, nrow * 2.0))
    for ax in np.atleast_1d(axes).ravel():
        ax.axis('off')
    for ax, s in zip(np.atleast_1d(axes).ravel(), stems):
        ax.imshow(load_xray(s), cmap='gray')
        y0, y1, x0, x1 = ROI_BOX[s]
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec='r', lw=1.2))
        ax.set_title(s[8:], fontsize=7)
    fig.suptitle(title); plt.tight_layout(); plt.show()

rng = np.random.RandomState(0)
show_boxes(list(rng.choice(x_stems, 24, replace=False)), title='Boîtes de recadrage (échantillon) — vérifiez')
if NEED_ROI:
    fig, axes = plt.subplots(2, 12, figsize=(14.4, 4.5))
    for ax, s in zip(axes.ravel(), rng.choice(x_stems, 24, replace=False)):
        ax.imshow(load_roi(s), cmap='gray'); ax.axis('off'); ax.set_title(s[8:], fontsize=7)
    fig.suptitle('Recadrages haute résolution'); plt.tight_layout(); plt.show()

## 7. Piste C : détection du matériel chirurgical (vis, tiges, ciment)
Le métal (ou le ciment de vertébroplastie) apparaît **saturé** et **beaucoup plus clair que son voisinage**. On mesure la fraction de ces pixels **dans la boîte du rachis** (pour ignorer les prothèses de hanche et les marqueurs), puis on prend le maximum sur les vues du patient.

Vérifiez la mosaïque, puis corrigez si besoin avec `HW_FORCE_YES` / `HW_FORCE_NO` (numéros de patients).

In [ ]:
HW_FORCE_YES, HW_FORCE_NO = set(), set()     # ex. {123, 456}

def metal_map(a):
    b = a.astype(np.float32)
    return (b >= 250) & ((b - ndi.gaussian_filter(b, 8)) > 30)

def metal_score(a, box):
    y0, y1, x0, x1 = [int(round(v)) for v in box]
    m = metal_map(a)[max(y0, 0):y1, max(x0, 0):x1]
    return float(m.mean()) if m.size else 0.0

hw_file = CACHE / 'hardware_scores.csv'
if hw_file.exists():
    hw_img = pd.read_csv(hw_file)
else:
    hw_img = pd.DataFrame(dict(stem=x_stems, metal=[metal_score(load_xray(s), ROI_BOX[s]) for s in tqdm(x_stems, desc='Score métal')]))
    hw_img.to_csv(hw_file, index=False)
score_of = dict(zip(hw_img.stem, hw_img.metal))
df['metal_score'] = [max([score_of[s] for s in L], default=0.0) for L in df.xray_stems]
df['has_hw'] = (df.metal_score > HW_THR).astype(float)
df.loc[df.patient_id.isin(HW_FORCE_YES), 'has_hw'] = 1.0
df.loc[df.patient_id.isin(HW_FORCE_NO), 'has_hw'] = 0.0
HAS_HW = df['has_hw'].values.astype(bool)
print(f'Patients avec matériel détecté : {HAS_HW.sum()} / {len(df)} ({HAS_HW.mean() * 100:.1f} %)')
display(pd.crosstab(df.has_hw.map({0: 'sans matériel', 1: 'avec matériel'}), df.Osteoporosis.map(dict(enumerate(CLASSES))), normalize='index').round(2))

def best_view(L):
    return max(L, key=lambda s: score_of[s]) if L else None
hw_pids = df[HAS_HW].sort_values('metal_score').xray_stems.apply(best_view).dropna().tolist()
show_mosaic(hw_pids[:24], title='Matériel détecté — les plus proches du seuil (vérifiez)')
near = df[~HAS_HW].sort_values('metal_score', ascending=False).xray_stems.apply(best_view).dropna().tolist()
show_mosaic(near[:12], title='NON détectés mais proches du seuil (vérifiez)')

# Masques de tokens métalliques (option « mask ») : un token est masqué si > 3 % de sa surface est métallique
def token_metal_mask(a, thr=0.03):
    m = metal_map(a).astype(np.float32)
    gh, gw = TOKEN_GRID
    t = torch.from_numpy(m)[None, None]
    frac = F.adaptive_avg_pool2d(t, TOKEN_GRID)[0, 0].numpy()
    return frac <= thr                                   # True = token conservé

TOKMASK = {}
for roi, loader in (('full', load_xray),) + ((('spine', load_roi),) if NEED_ROI else ()):
    f = CACHE / f'tokmask_{roi}.npy'
    if f.exists():
        TOKMASK[roi] = np.load(f)
    else:
        mk = np.stack([token_metal_mask(loader(s)) for s in tqdm(x_stems, desc=f'Masques métal {roi}')])
        mk = np.stack([mk, mk[:, :, ::-1]], 1).reshape(len(x_stems), 2, -1)      # orig / miroir
        np.save(f, mk); TOKMASK[roi] = mk
print('Tokens masqués (moyenne, patients avec matériel) :',
      f"{(~TOKMASK['full'][[X_IDX[s] for L in df[HAS_HW].xray_stems for s in L]]).mean() * 100:.1f} %")

## 8. Tokens d'images pour chaque configuration (ACP sans étiquettes)
Les caractéristiques de chaque couple (extracteur, région) sont calculées une seule fois puis mises en cache. Chaque token est ensuite projeté sur ses `PCA_DIM` premières composantes principales (blanchiment). L'ACP n'utilise **aucune étiquette** : pas de fuite. `PCA_DIM = None` garde toutes les dimensions (plus lent).

In [ ]:
PCA_DIM = 384
FH, FW = TOKEN_GRID
N_TOK = FH * FW
N_img = len(x_stems)

def fit_token_pca(X, k):
    X = X.float(); mu = X.mean(0)
    evals, evecs = torch.linalg.eigh(torch.cov((X - mu).T))
    return mu, evecs[:, -k:].flip(1) / evals[-k:].flip(0).clamp_min(1e-6).sqrt()

_TOKENS = {}
def get_tokens(bb, roi):
    """Tokens (N_img, 2, 160, d) float16 pour un extracteur et une région ('full' ou 'spine')."""
    if (bb, roi) in _TOKENS:
        return _TOKENS[(bb, roi)]
    extract_features(bb, roi, load_xray if roi == 'full' else load_roi)
    XF = np.load(feat_file(bb, roi), mmap_mode='r')
    C = XF.shape[2]
    def tok(i0, i1):
        return torch.from_numpy(np.ascontiguousarray(XF[i0:i1])).permute(0, 1, 3, 4, 2).reshape(i1 - i0, 2, N_TOK, C)
    if PCA_DIM and PCA_DIM < C:
        rs = np.random.RandomState(0)
        samp = torch.cat([tok(i, i + 1).reshape(-1, C) for i in sorted(rs.choice(N_img, 150, replace=False))])
        mu, W = fit_token_pca(samp, PCA_DIM); del samp
        out = torch.empty(N_img, 2, N_TOK, PCA_DIM, dtype=torch.float16)
        for i in range(0, N_img, 64):
            j = min(i + 64, N_img)
            out[i:j] = ((tok(i, j).float() - mu) @ W).half()
    else:
        out = torch.cat([tok(i, min(i + 64, N_img)) for i in range(0, N_img, 64)])
    _TOKENS[(bb, roi)] = out
    return out

mu_c, W_c = fit_token_pca(CTF.reshape(-1, CTF.shape[-1]), PCA_DIM or CTF.shape[-1])
CTT = ((CTF.float() - mu_c) @ W_c).half()                      # tokens CT (N_ct, 16, d)
D_CT = CTT.shape[-1]
print('Tokens CT :', tuple(CTT.shape))

## 9. Le modèle PRISM

```
 Vues (AP, LAT, et/ou recadrages rachis) ─► tokens spatiaux (+ position, + type de vue, masque métal)
       ─► attention spatiale ─► attention sur les vues ─► FiLM(clinique) ─► e_x
 CT : 16 coupes ─► attention sur les coupes ─┐
 CT : HU trabéculaires ─► MLP ───────────────┴─► e_ct
 Clinique (5 var. [+ matériel]) ─► MLP ─► e_clin
 [e_x, e_ct, e_clin] + masques de présence ─► fusion par attention ─► h
 h ─► CORN (3 classes ordinales) | T-score | 44 mesures DXA (cibles privilégiées)
```

In [ ]:
class GatedAttnPool(nn.Module):
    """Agrégation par attention à porte (Ilse et al., 2018) avec masque."""
    def __init__(self, d, h=128):
        super().__init__()
        self.V, self.U, self.w = nn.Linear(d, h), nn.Linear(d, h), nn.Linear(h, 1)
    def forward(self, x, mask=None):
        a = self.w(torch.tanh(self.V(x)) * torch.sigmoid(self.U(x))).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(~mask, -1e4)
        w = torch.softmax(a, dim=1)
        return (w.unsqueeze(-1) * x).sum(1), w

def mlp(i, o, p=0.2):
    return nn.Sequential(nn.Linear(i, o), nn.GELU(), nn.Dropout(p), nn.Linear(o, o))

class PRISM(nn.Module):
    def __init__(self, d_tok, n_tok, n_slices, n_priv, n_clin, d=D_MODEL, n_hu=len(HU_COLS), p=0.2, d_ct=None, n_view_types=4,
                 clin_enc='mlp', film=True):
        super().__init__()
        d_ct = d_ct or d_tok
        self.x_proj = nn.Sequential(nn.LayerNorm(d_tok), nn.Linear(d_tok, d), nn.GELU(), nn.Dropout(p))
        self.x_pos  = nn.Parameter(torch.zeros(1, n_tok, d))
        self.x_pool = GatedAttnPool(d)
        self.view_emb = nn.Embedding(n_view_types, d)
        self.v_pool = GatedAttnPool(d)
        self.ct_proj = nn.Sequential(nn.LayerNorm(d_ct), nn.Linear(d_ct, d), nn.GELU(), nn.Dropout(p))
        self.ct_pos  = nn.Parameter(torch.zeros(1, n_slices, d))
        self.ct_pool = GatedAttnPool(d)
        self.hu_mlp  = mlp(n_hu, d, p)
        # encodeur clinique : 'mlp' (5 → 256 → 256, défaut) ou 'linear' (projection 5 → 256) — cf. section 18
        self.clin_mlp = mlp(n_clin, d, p) if clin_enc == 'mlp' else nn.Linear(n_clin, d)
        self.film = nn.Linear(d, 2 * d) if film else None
        self.mod_emb = nn.Parameter(torch.zeros(1, 3, d))
        self.m_pool = GatedAttnPool(d)
        self.trunk = nn.Sequential(nn.LayerNorm(2 * d), nn.Linear(2 * d, d), nn.GELU(), nn.Dropout(p))
        self.head_corn = nn.Linear(d, 2)
        self.head_t    = nn.Linear(d, 1)
        self.head_priv = nn.Linear(d, n_priv)
        nn.init.normal_(self.x_pos, std=0.02); nn.init.normal_(self.ct_pos, std=0.02); nn.init.normal_(self.mod_emb, std=0.02)

    def encode(self, b, tok_drop=0.0):
        B, V, N, _ = b['xtok'].shape
        t = self.x_proj(b['xtok']).reshape(B * V, N, -1) + self.x_pos
        m = b['tokmask'].reshape(B * V, N)
        if tok_drop > 0:
            m = m & (torch.rand(B * V, N, device=t.device) > tok_drop)
        m = m | ~m.any(1, keepdim=True)                                 # jamais tout masquer
        ex, a_sp = self.x_pool(t, m)
        ex = ex.reshape(B, V, -1) + self.view_emb(b['view'])
        xm = b['xmask']
        ex, a_v = self.v_pool(ex, xm | ~xm.any(1, keepdim=True))
        ec = self.clin_mlp(b['clin'])
        if self.film is not None:                                       # FiLM : la clinique module l'image
            g, be = self.film(ec).chunk(2, -1)
            ex = ex * (1 + g) + be
        ct = self.ct_proj(b['cttok']) + self.ct_pos
        ect, a_ct = self.ct_pool(ct)
        ect = ect + self.hu_mlp(b['hu'])
        return dict(ex=ex, ect=ect, ec=ec, a_sp=a_sp.reshape(B, V, N), a_v=a_v, a_ct=a_ct)

    def fuse(self, e, mods_mask):
        M = torch.stack([e['ex'], e['ect'], e['ec']], 1) + self.mod_emb
        h, a_m = self.m_pool(M, mods_mask)
        h = self.trunk(torch.cat([h, e['ec']], -1))
        return dict(corn=self.head_corn(h), t=self.head_t(h).squeeze(-1), priv=self.head_priv(h), a_m=a_m)

def corn_probs(logits):
    s1 = torch.sigmoid(logits[:, 0]); s2 = s1 * torch.sigmoid(logits[:, 1])
    return torch.stack([1 - s1, s1 - s2, s2], 1).clamp(1e-6, 1)

def corn_loss(logits, y):
    l = F.binary_cross_entropy_with_logits(logits[:, 0], (y > 0).float())
    sel = y > 0
    if sel.any():
        l = l + F.binary_cross_entropy_with_logits(logits[sel, 1], (y[sel] > 1).float())
    return l

### 9.1 Données par patient et configuration d'une expérience

In [ ]:
N_PAT = len(df)
PAT_X1 = np.full((N_PAT, 2), -1); PAT_V1 = np.zeros((N_PAT, 2), int)
for i, L in enumerate(df.xray_stems):
    for j, s in enumerate(L[:2]):
        PAT_X1[i, j] = X_IDX[s]; PAT_V1[i, j] = VIEW_OF[s]
PAT_CT = np.array([CT_IDX.get(s, -1) if isinstance(s, str) else -1 for s in df.ct_stem])
Y = df['Osteoporosis'].values; T = df['t_value'].values.astype(np.float32)
HU_RAW, PRIV_RAW = df[HU_COLS].values.astype(np.float32), df[PRIV].values.astype(np.float32)

EXP = {}
def set_experiment(bb, roi, hw):
    """Prépare les tokens, les vues, les entrées cliniques et les exclusions d'une expérience."""
    global XT, D_TOK, PAT_X, PAT_VIEW, TOKM, CLIN_USED, CLIN_RAW, TRAIN_EXCLUDE
    parts = ['full', 'spine'] if roi == 'full+spine' else [roi]
    XT = torch.cat([get_tokens(bb, r) for r in parts])                       # (N_img*len(parts), 2, 160, d)
    TOKM = torch.from_numpy(np.concatenate([TOKMASK[r] for r in parts]) if hw == 'mask'
                            else np.ones((len(parts) * N_img, 2, N_TOK), bool))
    PAT_X = np.concatenate([np.where(PAT_X1 >= 0, PAT_X1 + k * N_img, -1) for k in range(len(parts))], 1)
    PAT_VIEW = np.concatenate([PAT_V1 + 2 * (r == 'spine') for r in parts], 1)   # 0/1 = AP/LAT entières, 2/3 = recadrées
    D_TOK = XT.shape[-1]
    CLIN_USED = CLIN + (['has_hw'] if hw == 'flag' else [])
    CLIN_RAW = df[CLIN_USED].values.astype(np.float32)
    TRAIN_EXCLUDE = HAS_HW.copy() if hw == 'exclude_train' else np.zeros(N_PAT, bool)
    EXP.update(bb=bb, roi=roi, hw=hw)

def fold_stats(tr):
    def ms(A):
        return np.nanmean(A[tr], 0), np.nanstd(A[tr], 0) + 1e-6
    return dict(clin=ms(CLIN_RAW), hu=ms(HU_RAW), priv=ms(PRIV_RAW))

def make_batch(idx, st, train, flip=0, with_xtok=True):
    idx = np.asarray(idx); B = len(idx)
    xi = PAT_X[idx]; xs = np.clip(xi, 0, None)
    fl = np.random.randint(0, 2, size=xi.shape) if train else np.full(xi.shape, flip)
    xtok = XT[xs, fl].float() if with_xtok else torch.zeros(B, xi.shape[1], 1, 1)
    tokmask = TOKM[xs, fl] | torch.from_numpy(xi < 0)[..., None]
    ci = PAT_CT[idx]
    hu = np.nan_to_num((HU_RAW[idx] - st['hu'][0]) / st['hu'][1])
    clin = (CLIN_RAW[idx] - st['clin'][0]) / st['clin'][1]
    pv = (PRIV_RAW[idx] - st['priv'][0]) / st['priv'][1]
    b = dict(xtok=xtok, tokmask=tokmask, xmask=torch.from_numpy(xi >= 0), view=torch.from_numpy(PAT_VIEW[idx]),
             cttok=CTT[np.clip(ci, 0, None)].float(), ctmask=torch.from_numpy(ci >= 0),
             hu=torch.from_numpy(hu).float(), clin=torch.from_numpy(clin).float(),
             y=torch.from_numpy(Y[idx]).long(), t=torch.from_numpy(T[idx]),
             priv=torch.from_numpy(np.nan_to_num(pv)).float(), priv_m=torch.from_numpy(~np.isnan(pv)),
             flip=torch.from_numpy(fl))
    return {k: v.to(DEVICE) for k, v in b.items()}

def modality_mask(b, mods, train, drop_ct=False):
    B = b['y'].shape[0]
    hx = b['xmask'].any(1) & ('xray' in mods)
    hc = b['ctmask'] & ('ct' in mods) & (not drop_ct)
    hl = torch.full((B,), 'clin' in mods, device=DEVICE)
    if train:
        dx = torch.rand(B, device=DEVICE) < P_DROP_XRAY
        hx = hx & ~(dx & (hc | hl))
        hc = hc & ~(torch.rand(B, device=DEVICE) < P_DROP_CT)
    m = torch.stack([hx, hc, hl], 1)
    m[~m.any(1), 2] = True
    return m

## 10. Entraînement (extracteur gelé ou fine-tuning) et validation croisée
Pour chaque pli externe : 85 % des plis d'entraînement servent à apprendre, 15 % (stratifiés) à l'arrêt précoce et au réglage des seuils ordinaux. Le pli de test n'est jamais vu. Avec `exclude_train`, les patients avec matériel sont retirés de l'entraînement et de la validation interne, mais **restent dans le test**.

**Fonction de perte** : `CORN + λ_T · SmoothL1(T-score) + λ_priv · MSE(44 mesures DXA) + λ_distill · [KL(p_sansCT ‖ p_avecCT) + MSE(T_sansCT, T_avecCT)]`

**Fine-tuning (piste D)** : l'extracteur est ré-entraîné avec la tête PRISM (taux d'apprentissage 2e-5 pour l'extracteur, 3e-4 pour la tête), avec augmentations (rotation ±7°, translation, zoom, contraste, miroir ; sans déformation géométrique quand l'option `mask` est active, pour que les masques restent alignés).

In [ ]:
from sklearn.model_selection import train_test_split
import torchvision.transforms.v2 as T2
AUG_GEO = T2.RandomAffine(degrees=7, translate=(0.05, 0.05), scale=(0.9, 1.1))
AUG_INT = T2.ColorJitter(brightness=0.2, contrast=0.2)
_IMG = {}

def image_batch(idx, flips, train):
    if not _IMG:
        _IMG['full'] = {s: load_xray(s) for s in tqdm(x_stems, desc='Images entières en mémoire')}
        if NEED_ROI:
            _IMG['spine'] = {s: load_roi(s) for s in tqdm(x_stems, desc='Recadrages en mémoire')}
    parts = ['full', 'spine'] if EXP['roi'] == 'full+spine' else [EXP['roi']]
    xi = PAT_X[np.asarray(idx)]
    imgs = torch.zeros(xi.shape[0], xi.shape[1], 1, XRAY_H, XRAY_W)
    for a in range(xi.shape[0]):
        for v in range(xi.shape[1]):
            if xi[a, v] < 0:
                continue
            k, s = divmod(int(xi[a, v]), N_img)
            t = torch.from_numpy(_IMG[parts[k]][x_stems[s]]).float().div(255)[None]
            if train:
                t = AUG_INT(t)
                if EXP['hw'] != 'mask':
                    t = AUG_GEO(t)
            if flips[a, v]:
                t = t.flip(-1)
            imgs[a, v] = t
    return imgs.to(DEVICE)

class E2E(nn.Module):
    """Extracteur entraînable + tête PRISM."""
    def __init__(self, bb):
        super().__init__()
        self.bb = Backbone(bb)
        with torch.no_grad():
            C = self.bb.eval()(torch.zeros(1, 1, XRAY_H, XRAY_W)).shape[1]
        self.bb.prepare_finetune()
        self.head = PRISM(C, N_TOK, CT_SLICES, len(PRIV), len(CLIN_USED), d_ct=D_CT)
    def train(self, mode=True):
        super().train(mode)
        for m in self.bb.modules():                   # BatchNorm figée : stable avec de petits micro-lots
            if isinstance(m, nn.modules.batchnorm._BatchNorm):
                m.eval()
        return self
    def encode(self, b, tok_drop=0.0):
        imgs = b['imgs']; B, V = imgs.shape[:2]
        f = self.bb(imgs.reshape(B * V, 1, XRAY_H, XRAY_W))
        return self.head.encode(dict(b, xtok=f.flatten(2).transpose(1, 2).reshape(B, V, N_TOK, -1)), tok_drop)
    def fuse(self, e, mm):
        return self.head.fuse(e, mm)

def get_batch(idx, st, train, ft, flip=0):
    b = make_batch(idx, st, train=train, flip=flip, with_xtok=not ft)
    if ft:
        b['imgs'] = image_batch(idx, b['flip'].cpu().numpy(), train)
    return b

def forward_all(model, b, mods, train, drop_ct=False, tok_drop=0.0):
    if 'clin' not in mods:
        b = dict(b, clin=torch.zeros_like(b['clin']))
    e = model.encode(b, tok_drop=tok_drop)
    mm = modality_mask(b, mods, train, drop_ct)
    return e, model.fuse(e, mm), mm

def compute_loss(model, b, cfg, train):
    e, o, mm = forward_all(model, b, cfg['mods'], train, tok_drop=0.1 if train else 0.0)
    loss = corn_loss(o['corn'], b['y']) + cfg.get('lambda_t', LAMBDA_T) * F.smooth_l1_loss(o['t'], b['t'])
    if cfg['priv']:
        pm = b['priv_m'].float()
        if cfg.get('priv_drop_t'):                    # ablation : aucune information de T-score / Z-score
            pm = pm * PRIV_KEEP_NO_T.to(pm.device)
        loss = loss + LAMBDA_PRIV * (((o['priv'] - b['priv']) ** 2) * pm).sum() / pm.sum().clamp_min(1)
    if cfg['distill'] and train:
        has = mm[:, 1]
        if has.any():
            mm2 = mm.clone(); mm2[:, 1] = False
            o2 = model.fuse(e, mm2)
            pt, ps = corn_probs(o['corn'][has]).detach(), corn_probs(o2['corn'][has])
            kl = (pt * (pt.log() - ps.log())).sum(1).mean()
            loss = loss + LAMBDA_DISTILL * (kl + F.mse_loss(o2['t'][has], o['t'][has].detach()))
    return loss

@torch.no_grad()
def predict(model, idx, st, cfg, ft=False, drop_ct=False, tta=True):
    model.eval(); P, Tp = [], []
    bs = FT_BATCH if ft else 128
    for i in range(0, len(idx), bs):
        pp, tt, flips = 0, 0, ((0, 1) if tta else (0,))
        for flip in flips:
            b = get_batch(idx[i:i + bs], st, False, ft, flip)
            _, o, _ = forward_all(model, b, cfg['mods'], False, drop_ct=drop_ct)
            pp = pp + corn_probs(o['corn']) / len(flips); tt = tt + o['t'] / len(flips)
        P.append(pp.cpu()); Tp.append(tt.cpu())
    return torch.cat(P).numpy(), torch.cat(Tp).numpy()

def train_one(tr_idx, va_idx, cfg, seed, ft=False):
    seed_everything(seed)
    st = fold_stats(tr_idx)
    if ft:
        model = E2E(EXP['bb']).to(DEVICE)
        opt = torch.optim.AdamW([{'params': [p for p in model.bb.parameters() if p.requires_grad], 'lr': FT_LR_BB},
                                 {'params': model.head.parameters(), 'lr': FT_LR_HEAD}], weight_decay=WD)
        n_ep, pat, bs, sched = FT_EPOCHS, FT_PATIENCE, FT_BATCH, None
    else:
        model = PRISM(D_TOK, N_TOK, CT_SLICES, len(PRIV), len(CLIN_USED), d_ct=D_CT,
                      clin_enc=cfg.get('clin_enc', 'mlp'), film=cfg.get('film', True)).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        n_ep, pat, bs = N_EPOCHS, PATIENCE, BATCH
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=n_ep * int(math.ceil(len(tr_idx) / bs)), pct_start=0.1)
    best, best_state, wait = 1e9, None, 0
    for ep in range(n_ep):
        t_ep = time.time()
        model.train()
        perm = np.random.permutation(tr_idx)
        for i in tqdm(range(0, len(perm), bs), desc=f'époque {ep}', leave=False, disable=not ft):
            chunk = perm[i:i + bs]
            opt.zero_grad()
            micro = FT_MICRO if ft else len(chunk)          # accumulation de gradients (mémoire GPU)
            for j in range(0, len(chunk), micro):
                mb = chunk[j:j + micro]
                loss = compute_loss(model, get_batch(mb, st, True, ft), cfg, True) * (len(mb) / len(chunk))
                loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if sched is not None:
                sched.step()
        if ft:
            print(f'    époque {ep + 1}/{n_ep} : {(time.time() - t_ep) / 60:.1f} min', flush=True)
            if (ep + 1) % FT_VAL_EVERY and ep + 1 < n_ep:
                continue
        pv, tv = predict(model, va_idx, st, cfg, ft=ft, tta=False)
        vl = F.nll_loss(torch.log(torch.tensor(pv)), torch.tensor(Y[va_idx])).item() + LAMBDA_T * float(np.mean(np.abs(tv - T[va_idx])))
        if vl < best - 1e-4:
            best, wait = vl, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= pat:
                break
    model.load_state_dict(best_state)
    del opt
    free_gpu_memory()
    return model, st, ep + 1

def fit_cutpoints(score, y):
    """Seuils ordinaux sur le score espéré s = p1 + 2·p2, choisis pour maximiser le QWK (validation interne)."""
    grid = np.linspace(0.2, 1.8, 81); best = (-1, 0.5, 1.5)
    for c1 in grid:
        for c2 in grid[grid > c1]:
            q = cohen_kappa_score(y, np.digitize(score, [c1, c2]), weights='quadratic')
            if q > best[0]:
                best = (q, c1, c2)
    return best[1:]

def run_cv(cfg, seeds, name, ft=False):
    S = len(seeds)
    oof_p_seed = np.zeros((S, N_PAT, 3)); oof_t_seed = np.zeros((S, N_PAT))
    oof_cls = np.zeros((N_PAT, S), int); oof_p_noct = np.zeros((N_PAT, 3)); oof_t_noct = np.zeros(N_PAT)
    val_qwk = np.zeros((S, len(FOLDS)))                       # QWK sur la validation interne (sélection imbriquée)
    models, t0 = [], time.time()
    for si, seed in enumerate(seeds):
        for k in FOLDS:
            tr_all = np.where((df.fold != k).values & ~TRAIN_EXCLUDE)[0]; te = np.where(df.fold == k)[0]
            tr, va = train_test_split(tr_all, test_size=0.15, stratify=Y[tr_all], random_state=seed)
            if SMOKE and ft:
                tr, va = tr[:8], va[:4]
            part = OUT / f"ftpart_{name.replace(' ', '').replace('|', '_')}_s{seed}_f{k}{'_smoke' if SMOKE else ''}.pt"
            if ft and part.exists():                         # fine-tuning : reprise pli par pli
                d = torch.load(part, map_location='cpu', weights_only=False)
                p, t, pv, c1, c2, p2, t2 = d['p'], d['t'], d['pv'], d['c1'], d['c2'], d['p2'], d['t2']
                print(f'  [{name}] graine {seed} pli {k} : rechargé'); 
                oof_p_seed[si, te], oof_t_seed[si, te] = p, t
                oof_cls[te, si] = np.digitize(p[:, 1] + 2 * p[:, 2], [c1, c2])
                va_saved = d.get('va', va)
                val_qwk[si, FOLDS.index(k)] = cohen_kappa_score(Y[va_saved], pv.argmax(1), weights='quadratic')
                if 'ct' in cfg['mods'] and p2 is not None:
                    oof_p_noct[te] += p2 / S; oof_t_noct[te] += t2 / S
                continue
            model, st, n_ep = train_one(tr, va, cfg, seed, ft)
            p, t = predict(model, te, st, cfg, ft=ft)
            pv, _ = predict(model, va, st, cfg, ft=ft)
            c1, c2 = fit_cutpoints(pv[:, 1] + 2 * pv[:, 2], Y[va])
            val_qwk[si, FOLDS.index(k)] = cohen_kappa_score(Y[va], pv.argmax(1), weights='quadratic')
            oof_p_seed[si, te], oof_t_seed[si, te] = p, t
            oof_cls[te, si] = np.digitize(p[:, 1] + 2 * p[:, 2], [c1, c2])
            p2, t2 = (predict(model, te, st, cfg, ft=ft, drop_ct=True) if 'ct' in cfg['mods'] else (None, None))
            if p2 is not None:
                oof_p_noct[te] += p2 / S; oof_t_noct[te] += t2 / S
            if ft:
                torch.save(dict(p=p, t=t, pv=pv, va=va, c1=c1, c2=c2, p2=p2, t2=t2), part)
            if not ft:
                models.append(dict(seed=seed, fold=k, stats=st, cut=(c1, c2),
                                   state={kk: v.half() for kk, v in model.state_dict().items()}))
            print(f'  [{name}] graine {seed} pli {k} : {n_ep} époques, QWK pli = '
                  f'{cohen_kappa_score(Y[te], p.argmax(1), weights="quadratic"):.3f}  ({time.time() - t0:.0f} s)')
            del model
            free_gpu_memory()
    return dict(p=oof_p_seed.mean(0), t=oof_t_seed.mean(0), p_seed=oof_p_seed, t_seed=oof_t_seed,
                cls_ord=np.array([np.bincount(r, minlength=3).argmax() for r in oof_cls]),
                p_noct=oof_p_noct, t_noct=oof_t_noct, models=models, seeds=list(seeds), pca=PCA_DIM, val_qwk=val_qwk)

CONFIGS = {
    'Clinique (MLP)':                 dict(mods={'clin'},               priv=False, distill=False),
    'Radiographie seule':             dict(mods={'xray'},               priv=True,  distill=False),
    'Radiographie + clinique':        dict(mods={'xray', 'clin'},       priv=True,  distill=False),
    'PRISM sans info privilégiée':    dict(mods={'xray', 'ct', 'clin'}, priv=False, distill=True),
    'PRISM sans auto-distillation':   dict(mods={'xray', 'ct', 'clin'}, priv=True,  distill=False),
    'PRISM sans régression du T-score': dict(mods={'xray', 'ct', 'clin'}, priv=True, distill=True, lambda_t=0.0, priv_drop_t=True),
    'PRISM (complet)':                dict(mods={'xray', 'ct', 'clin'}, priv=True,  distill=True),
}
# colonnes DXA autorisées quand on retire toute information de T-score / Z-score
PRIV_KEEP_NO_T = torch.tensor([0.0 if (c == 't_value' or c.startswith(('T_L', 'Z_L'))) else 1.0 for c in PRIV])
MAIN = 'PRISM (complet)'

## 11. Grille d'expériences
Chaque expérience = (extracteur, région, matériel, fine-tuning) avec le modèle **PRISM complet**. Les résultats sont enregistrés au fur et à mesure dans `results_prism/` : une exécution interrompue **reprend** là où elle s'était arrêtée.

**Durées indicatives** (Mac Apple Silicon, 3 graines) : environ 3 à 8 min par expérience gelée (le double pour `full+spine`), soit environ 2 à 4 h pour les 24 expériences de la grille par défaut. L'extraction des caractéristiques RAD-DINO (une seule fois) prend environ 15 à 30 min. Le fine-tuning est beaucoup plus long : 1 à 3 h par expérience et par graine pour `resnet50`, davantage pour `raddino`. D'où `FINETUNE_GRID = 'best'` par défaut.

In [ ]:
def exp_key(bb, roi, hw, ft):
    return (bb, roi, hw, bool(ft))

def exp_label(k):
    return f"{k[0]} | {k[1]} | {k[2]} | {'fine-tuné' if k[3] else 'gelé'}"

def ck_path(k, cfg_name=MAIN):
    tag = f"{k[0]}_{k[1].replace('+', '-')}_{k[2]}_{'ft' if k[3] else 'frozen'}_{cfg_name}"
    if CT_ACTIVE != 'resnet50' and not (k[3] and REUSE_FT_RESNET_CT):
        tag += f'_ct{CT_ACTIVE}'                      # résultats obtenus avec un autre encodeur CT : sauvegarde distincte
    tag = tag.replace(' ', '_').replace('(', '').replace(')', '')
    return OUT / f"exp_{tag}{'_smoke' if SMOKE else ''}.pt"

RUNS = {}
def run_experiment(k, cfg_name=MAIN, seeds=None):
    seeds = seeds or (FINETUNE_SEEDS if k[3] else GRID_SEEDS)
    ck = ck_path(k, cfg_name)
    if ck.exists():
        saved = torch.load(ck, map_location='cpu', weights_only=False)
        if saved.get('seeds') == list(seeds) and saved.get('pca') == PCA_DIM and 'val_qwk' in saved:
            RUNS[(k, cfg_name)] = saved
            print('✔', exp_label(k), '|', cfg_name, '(rechargé)')
            return saved
    try:
        set_experiment(*k[:3])
    except Exception as err:                       # ex. RAD-DINO non téléchargeable
        print('✖', exp_label(k), ': ignorée ->', repr(err)[:200])
        return None
    print('▶', exp_label(k), '|', cfg_name)
    ct_tag = f' | CT {CT_ACTIVE}' if CT_ACTIVE != 'resnet50' else ''
    r = run_cv(CONFIGS[cfg_name], seeds, f'{exp_label(k)} | {cfg_name}{ct_tag}', ft=k[3])
    torch.save(r, ck)
    RUNS[(k, cfg_name)] = r
    return r

FROZEN_EXPS = [exp_key(b, r, h, False) for b in GRID['backbone'] for r in GRID['roi'] for h in GRID['hardware']]
print(f'{len(FROZEN_EXPS)} expériences gelées à évaluer')
for k in FROZEN_EXPS:
    run_experiment(k)

### 11.1 Statistiques : intervalles de confiance et test de DeLong

In [ ]:
def delong_test(y_true, s1, s2):
    """Test de DeLong pour deux AUC corrélées (mêmes patients). Retourne (auc1, auc2, p-valeur bilatérale)."""
    from scipy.stats import norm
    y_true = np.asarray(y_true).astype(bool)
    pos, neg = np.where(y_true)[0], np.where(~y_true)[0]
    m, n = len(pos), len(neg)
    def comps(s):
        X, Yn = s[pos], s[neg]
        psi = (X[:, None] > Yn[None, :]) + 0.5 * (X[:, None] == Yn[None, :])
        return psi.mean(), psi.mean(1), psi.mean(0)
    a1, v10_1, v01_1 = comps(np.asarray(s1)); a2, v10_2, v01_2 = comps(np.asarray(s2))
    S10 = np.cov(np.vstack([v10_1, v10_2])); S01 = np.cov(np.vstack([v01_1, v01_2]))
    S = S10 / m + S01 / n
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    z = (a1 - a2) / np.sqrt(max(var, 1e-12))
    return a1, a2, 2 * (1 - norm.cdf(abs(z)))

def bootstrap_ci(y, prob, n_boot=1000, seed=0):
    rs = np.random.RandomState(seed); q, a, f = [], [], []
    pred = prob.argmax(1)
    for _ in range(n_boot):
        i = rs.randint(0, len(y), len(y))
        q.append(cohen_kappa_score(y[i], pred[i], weights='quadratic'))
        a.append(roc_auc_score(y[i] == 2, prob[i, 2])); f.append(f1_score(y[i], pred[i], average='macro'))
    ci = lambda v: tuple(np.percentile(v, [2.5, 97.5]))
    return dict(qwk_ci=ci(q), auc_osteo_ci=ci(a), f1_ci=ci(f))

def paired_qwk_pvalue(y, pa, pb, n_boot=1000, seed=0):
    """p-valeur (bootstrap apparié) pour la différence de QWK entre deux modèles."""
    rs = np.random.RandomState(seed); d = []
    for _ in range(n_boot):
        i = rs.randint(0, len(y), len(y))
        d.append(cohen_kappa_score(y[i], pa[i].argmax(1), weights='quadratic') - cohen_kappa_score(y[i], pb[i].argmax(1), weights='quadratic'))
    d = np.asarray(d)
    return float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))

def summarize(k, r, ref):
    m_all = metrics3(Y, r['p'], T, r['t'])
    nohw = ~HAS_HW
    m_nohw = metrics3(Y[nohw], r['p'][nohw], T[nohw], r['t'][nohw])
    ci = bootstrap_ci(Y, r['p'])
    row = dict(experience=exp_label(k), backbone=k[0], roi=k[1], hardware=k[2], finetune=k[3],
               qwk=m_all['qwk'], qwk_ci=f"[{ci['qwk_ci'][0]:.3f}–{ci['qwk_ci'][1]:.3f}]",
               f1_macro=m_all['f1_macro'], auc_macro=m_all['auc_macro'], auc_osteoporose=m_all['auc_osteoporose'],
               auc_osteo_ci=f"[{ci['auc_osteo_ci'][0]:.3f}–{ci['auc_osteo_ci'][1]:.3f}]",
               sens_osteoporose=m_all['sens_osteoporose'], spec_osteoporose=m_all['spec_osteoporose'],
               t_mae=m_all['t_mae'], qwk_sans_materiel=m_nohw['qwk'], auc_osteo_sans_materiel=m_nohw['auc_osteoporose'],
               qwk_sans_CT=metrics3(Y, r['p_noct'])['qwk'])
    if ref is not None and ref is not r:
        row['p_DeLong_vs_ref'] = delong_test(Y == 2, r['p'][:, 2], ref['p'][:, 2])[2]
        row['p_QWK_vs_ref'] = paired_qwk_pvalue(Y, r['p'], ref['p'])
    return row

def results_table():
    ref = RUNS.get((exp_key(*REFERENCE, False), MAIN))
    rows = [summarize(k, r, ref) for (k, c), r in RUNS.items() if c == MAIN]
    return pd.DataFrame(rows).sort_values('qwk', ascending=False).reset_index(drop=True)

grid_res = results_table()
grid_res.to_csv(OUT / 'grille_resultats.csv', index=False)
show_cols = ['experience', 'qwk', 'qwk_ci', 'f1_macro', 'auc_macro', 'auc_osteoporose', 'auc_osteo_ci', 'sens_osteoporose',
             'qwk_sans_materiel', 'qwk_sans_CT', 'p_DeLong_vs_ref', 'p_QWK_vs_ref']
display(grid_res[[c for c in show_cols if c in grid_res]].round(3))

### 11.2 Effet moyen de chaque option (moyenne marginale du QWK)

In [ ]:
for fac in ['backbone', 'roi', 'hardware']:
    if grid_res[fac].nunique() > 1:
        display(grid_res[~grid_res.finetune].groupby(fac)[['qwk', 'auc_osteoporose', 'qwk_sans_materiel']].mean().round(3)
                .sort_values('qwk', ascending=False))

## 12. Piste D : fine-tuning
### 12.0 Test pilote sur un seul pli (recommandé avant le fine-tuning complet)
Le fine-tuning complet (5 plis) peut prendre de nombreuses heures sur un Mac. Ce test entraîne **un seul pli** et compare le résultat au modèle gelé **sur ce même pli**. Si le gain est faible (< 0,02 de QWK), inutile de lancer les 5 plis.

Mettez `RUN_PILOT = True` et choisissez la configuration dans `PILOT_EXP`. Le fine-tuning complet de la section suivante reprend automatiquement le pli déjà calculé.

In [ ]:
RUN_PILOT = False
PILOT_EXP, PILOT_FOLD = ('raddino', 'spine', 'flag'), FOLDS[0]
if RUN_PILOT:
    kf = exp_key(*PILOT_EXP, True); kz = exp_key(*PILOT_EXP, False)
    set_experiment(*PILOT_EXP)
    tr_all = np.where((df.fold != PILOT_FOLD).values & ~TRAIN_EXCLUDE)[0]; te = np.where(df.fold == PILOT_FOLD)[0]
    tr, va = train_test_split(tr_all, test_size=0.15, stratify=Y[tr_all], random_state=FINETUNE_SEEDS[0])
    t0 = time.time()
    model, st, n_ep = train_one(tr, va, CONFIGS[MAIN], FINETUNE_SEEDS[0], ft=True)
    p, t = predict(model, te, st, CONFIGS[MAIN], ft=True)
    pv, _ = predict(model, va, st, CONFIGS[MAIN], ft=True)
    c1, c2 = fit_cutpoints(pv[:, 1] + 2 * pv[:, 2], Y[va])
    p2, t2 = predict(model, te, st, CONFIGS[MAIN], ft=True, drop_ct=True)
    name = f'{exp_label(kf)} | {MAIN}'
    torch.save(dict(p=p, t=t, pv=pv, va=va, c1=c1, c2=c2, p2=p2, t2=t2),
               OUT / f"ftpart_{name.replace(' ', '').replace('|', '_')}_s{FINETUNE_SEEDS[0]}_f{PILOT_FOLD}{'_smoke' if SMOKE else ''}.pt")
    del model; free_gpu_memory()
    rows = {'Fine-tuné': metrics3(Y[te], p)}
    if (kz, MAIN) in RUNS:
        rows['Gelé (même pli)'] = metrics3(Y[te], RUNS[(kz, MAIN)]['p'][te])
    display(pd.DataFrame(rows).T[['qwk', 'f1_macro', 'auc_macro', 'auc_osteoporose', 'sens_osteoporose']].round(3))
    print(f'Durée du pilote : {(time.time() - t0) / 3600:.1f} h ({n_ep} époques) -> 5 plis ≈ {5 * (time.time() - t0) / 3600:.0f} h')

### 12.1 Fine-tuning complet
`FINETUNE_GRID = 'best'` fine-tune les `FINETUNE_TOP` meilleures configurations gelées. `'all'` fine-tune toutes les combinaisons (très long). `'none'` saute cette étape.

In [ ]:
if FINETUNE_GRID != 'none':
    frozen_sorted = [exp_key(r.backbone, r.roi, r.hardware, False) for r in grid_res[~grid_res.finetune].itertuples()]
    ft_list = frozen_sorted[:FINETUNE_TOP] if FINETUNE_GRID == 'best' else frozen_sorted
    for k in ft_list:
        run_experiment(exp_key(k[0], k[1], k[2], True))
    grid_res = results_table()
    grid_res.to_csv(OUT / 'grille_resultats.csv', index=False)
    display(grid_res[[c for c in show_cols if c in grid_res]].round(3))

### 12.2 Estimation imbriquée, sans biais de sélection
Choisir la meilleure de N configurations sur les mêmes données de test donne un score optimiste. Ici, **dans chaque pli externe**, la configuration est choisie d'après son QWK sur la **validation interne** de ce pli (moyenne des graines), puis on prend ses prédictions sur le pli de test. Le pli de test n'intervient jamais dans le choix.

Le score obtenu évalue tout le pipeline « tester toutes les configurations et garder la meilleure ». C'est le chiffre honnête à rapporter.

In [ ]:
if RUN_NESTED:
    cands = [(k, r) for (k, c), r in RUNS.items() if c == MAIN and 'val_qwk' in r]
    p_nest, t_nest, chosen = np.zeros((N_PAT, 3)), np.zeros(N_PAT), []
    for fi, k in enumerate(FOLDS):
        scores = [r['val_qwk'][:, fi].mean() for _, r in cands]
        kb, rb = cands[int(np.argmax(scores))]
        te = (df.fold == k).values
        p_nest[te], t_nest[te] = rb['p'][te], rb['t'][te]
        chosen.append(dict(pli=int(k), configuration=exp_label(kb), qwk_validation_interne=float(np.max(scores)),
                           qwk_test_du_pli=cohen_kappa_score(Y[te], rb['p'][te].argmax(1), weights='quadratic')))
    display(pd.DataFrame(chosen).round(3))
    m_nest = metrics3(Y, p_nest, T, t_nest); ci = bootstrap_ci(Y, p_nest)
    best_naive = grid_res.iloc[0]
    print(f"Sélection imbriquée  : QWK = {m_nest['qwk']:.3f} [{ci['qwk_ci'][0]:.3f}–{ci['qwk_ci'][1]:.3f}] | "
          f"AUC ostéoporose = {m_nest['auc_osteoporose']:.3f} [{ci['auc_osteo_ci'][0]:.3f}–{ci['auc_osteo_ci'][1]:.3f}]")
    print(f"Meilleure config. (choix « naïf » sur le test) : QWK = {best_naive.qwk:.3f}  ->  optimisme ≈ {best_naive.qwk - m_nest['qwk']:+.3f}")
    RESULTS['Sélection imbriquée (sans biais)'] = m_nest
    pd.DataFrame(chosen).to_csv(OUT / 'selection_imbriquee.csv', index=False)

## 13. Ablation des modalités sur la meilleure configuration gelée

In [ ]:
BEST = exp_key(*grid_res[~grid_res.finetune].iloc[0][['backbone', 'roi', 'hardware']], False)
if FINAL_MODEL and (exp_key(*FINAL_MODEL, False), MAIN) in RUNS:
    BEST = exp_key(*FINAL_MODEL, False)               # modèle final choisi à l'avance (cohérence de l'article)
print('Configuration utilisée pour les ablations :', exp_label(BEST))
abl_rows = {}
if RUN_ABLATIONS:
    for cfg_name in CONFIGS:
        r = run_experiment(BEST, cfg_name, GRID_SEEDS) if cfg_name != MAIN else RUNS[(BEST, MAIN)]
        if r is not None:
            abl_rows[cfg_name] = metrics3(Y, r['p'], T, r['t'])
    abl_rows['Référence clinique (LR)'] = RESULTS['Référence clinique (LR)']
    abl_rows['Sonde linéaire (image + clinique)'] = RESULTS['Sonde linéaire (image + clinique)']
    abl_rows[MAIN + ' — déployé SANS CT'] = metrics3(Y, RUNS[(BEST, MAIN)]['p_noct'], T, RUNS[(BEST, MAIN)]['t_noct'])
    abl = pd.DataFrame(abl_rows).T
    abl.to_csv(OUT / 'ablation_meilleure_config.csv')
    cols = ['qwk', 'f1_macro', 'bal_acc', 'acc', 'auc_macro', 'auc_osteoporose', 'sens_osteoporose', 'spec_osteoporose', 't_mae', 't_spearman']
    display(abl[[c for c in cols if c in abl]].round(3))

## 14. Figures de la meilleure configuration

In [ ]:
import seaborn as sns
from sklearn.metrics import roc_curve
best_row = grid_res.iloc[0]
BEST_ALL = exp_key(best_row.backbone, best_row.roi, best_row.hardware, best_row.finetune)
if FINAL_MODEL and (exp_key(*FINAL_MODEL, False), MAIN) in RUNS:
    BEST_ALL = exp_key(*FINAL_MODEL, False)
print('Modèle final utilisé pour les figures :', exp_label(BEST_ALL))
r = RUNS[(BEST_ALL, MAIN)]
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
sns.heatmap(confusion_matrix(Y, r['p'].argmax(1)), annot=True, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[0])
ax[0].set_xlabel('Prédit'); ax[0].set_ylabel('Réel'); ax[0].set_title(f'{exp_label(BEST_ALL)}\nmatrice de confusion (OOF)')
for c, col in zip(range(3), ['tab:green', 'tab:orange', 'tab:red']):
    fpr, tpr, _ = roc_curve(Y == c, r['p'][:, c])
    ax[1].plot(fpr, tpr, color=col, lw=2, label=f'{CLASSES[c]} (AUC = {roc_auc_score(Y == c, r["p"][:, c]):.3f})')
ax[1].plot([0, 1], [0, 1], 'k--', lw=1); ax[1].legend(); ax[1].set_title('ROC one-vs-rest (OOF)')
ax[1].set_xlabel('1 − spécificité'); ax[1].set_ylabel('Sensibilité')
ax[2].scatter(T, r['t'], c=Y, cmap='RdYlGn_r', s=12, alpha=.7)
lim = [min(T.min(), r['t'].min()) - .3, max(T.max(), r['t'].max()) + .3]
ax[2].plot(lim, lim, 'k--', lw=1); ax[2].axvline(-1, c='gray', ls=':'); ax[2].axvline(-2.5, c='gray', ls=':')
ax[2].set_xlabel('T-score réel (DXA)'); ax[2].set_ylabel('T-score prédit')
ax[2].set_title(f"T-score : Spearman = {spearmanr(T, r['t']).correlation:.3f}")
plt.tight_layout(); plt.savefig(OUT / 'figure_resultats.png', dpi=200); plt.show()

# QWK de chaque expérience avec IC 95 %
g = grid_res.iloc[::-1]
lo = g.qwk_ci.str.extract(r'\[([\d.]+)–([\d.]+)\]').astype(float)
fig, ax = plt.subplots(figsize=(9, 0.35 * len(g) + 1.5))
ax.errorbar(g.qwk, range(len(g)), xerr=[g.qwk - lo[0], lo[1] - g.qwk], fmt='o', color='tab:blue', ecolor='gray', capsize=3)
ax.axvline(RESULTS['Référence clinique (LR)']['qwk'], color='tab:red', ls='--', label='Référence clinique')
ax.set_yticks(range(len(g))); ax.set_yticklabels(g.experience, fontsize=8); ax.set_xlabel('QWK (IC 95 %)')
ax.legend(); plt.tight_layout(); plt.savefig(OUT / 'figure_grille_qwk.png', dpi=200); plt.show()

### 14.1 Où regarde le modèle ? (cartes d'attention, meilleure configuration gelée)

In [ ]:
set_experiment(*BEST[:3])
def attention_maps(i):
    m = [m for m in RUNS[(BEST, MAIN)]['models'] if m['fold'] == df.fold.iloc[i]][0]
    model = PRISM(D_TOK, N_TOK, CT_SLICES, len(PRIV), len(CLIN_USED), d_ct=D_CT).to(DEVICE)
    model.load_state_dict({k: v.float() for k, v in m['state'].items()}); model.eval()
    with torch.no_grad():
        return model.encode(make_batch([i], m['stats'], train=False))['a_sp'][0].reshape(-1, FH, FW).cpu().numpy()

parts = ['full', 'spine'] if BEST[1] == 'full+spine' else [BEST[1]]
rng = np.random.RandomState(0)
show = [rng.choice(np.where((Y == c) & (PAT_X1[:, 0] >= 0))[0]) for c in (0, 1, 2) for _ in range(2)]
nv = PAT_X.shape[1]
fig, axes = plt.subplots(nv, len(show), figsize=(3 * len(show), 4.5 * nv))
axes = np.atleast_2d(axes)
for j, i in enumerate(show):
    att = attention_maps(i)
    for v in range(nv):
        ax = axes[v, j]; ax.axis('off')
        if PAT_X[i, v] < 0:
            continue
        kpart, s = divmod(int(PAT_X[i, v]), N_img)
        img = (load_xray if parts[kpart] == 'full' else load_roi)(x_stems[s])
        ax.imshow(img, cmap='gray')
        ax.imshow(np.asarray(Image.fromarray((att[v] / att[v].max() * 255).astype(np.uint8)).resize(img.shape[::-1], Image.BICUBIC)), cmap='jet', alpha=0.35)
        if v == 0:
            ax.set_title(f'{CLASSES[Y[i]]}\nprédit : {CLASSES[RUNS[(BEST, MAIN)]["p"][i].argmax()]}', fontsize=9)
plt.tight_layout(); plt.savefig(OUT / 'figure_attention.png', dpi=150); plt.show()

In [ ]:
pred_df = df[['patient_id', 'fold', 'Osteoporosis', 't_value', 'has_hw']].copy()
pred_df[['p_normal', 'p_osteopenie', 'p_osteoporose']] = r['p']
pred_df['pred'] = r['p'].argmax(1); pred_df['pred_seuils'] = r['cls_ord']; pred_df['t_pred'] = r['t']
pred_df.to_csv(OUT / 'predictions_oof_meilleure_config.csv', index=False)
print('Résultats enregistrés dans', OUT)

## 16. Figures et tableaux pour l'article (IEEE Access)
Tout est exporté dans `results_prism/article/` : figures en **PDF vectoriel** et **PNG 300 dpi**, dimensionnées pour IEEE (une colonne = 3,5 in, deux colonnes = 7,16 in), tableaux en **CSV** et **LaTeX**. La langue est réglée par `LANG` (`'en'` par défaut).

Le modèle des figures est `FINAL_MODEL` (configuration). Mettez `FIG_SOURCE = 'nested'` pour utiliser les prédictions de la sélection imbriquée.

In [ ]:
FIG_SOURCE = 'best'                     # 'best' (= FINAL_MODEL) | 'nested'
ART = OUT / 'article'
ART.mkdir(exist_ok=True)
EN = LANG == 'en'
def L(fr, en):
    return en if EN else fr
CLS_ART = ['Normal', 'Osteopenia', 'Osteoporosis'] if EN else CLASSES
COL1, COL2 = 3.5, 7.16                  # largeurs IEEE (pouces)
plt.rcParams.update({'font.size': 8, 'axes.titlesize': 8.5, 'axes.labelsize': 8, 'legend.fontsize': 7,
                     'xtick.labelsize': 7.5, 'ytick.labelsize': 7.5, 'font.family': 'serif',
                     'font.serif': ['Times New Roman', 'Times', 'Nimbus Roman', 'DejaVu Serif'],
                     'axes.spines.top': False, 'axes.spines.right': False, 'savefig.bbox': 'tight', 'pdf.fonttype': 42})
CLS_COLORS = ['#2f7d5b', '#d08a1e', '#b8322a']            # normal, ostéopénie, ostéoporose
GREY = '#7a8794'

def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return ''
    return '< 0.001' if p < 0.001 else (f'{p:.3f}' if EN else f'{p:.3f}'.replace('.', ','))

def holm(pvals):
    # Correction de Holm-Bonferroni (comparaisons multiples)
    p = np.asarray(pvals, float); out = np.full_like(p, np.nan); ok = ~np.isnan(p)
    idx = np.argsort(p[ok]); m = ok.sum(); adj = np.empty(m); run = 0
    for r, i in enumerate(idx):
        run = max(run, min(1.0, (m - r) * p[ok][i])); adj[i] = run
    out[ok] = adj
    return out

def save_fig(fig, name):
    fig.savefig(ART / f'{name}.png', dpi=300)
    fig.savefig(ART / f'{name}.pdf')
    plt.show()

def save_table(tab, name, index=True):
    tab = tab.copy()
    for c in tab.columns:                       # colonnes numériques stockées en « object » -> nombres arrondis
        conv = pd.to_numeric(tab[c], errors='coerce')
        if conv.notna().sum() == tab[c].notna().sum() and tab[c].notna().any():
            tab[c] = conv.round(3)
    tab.to_csv(ART / f'{name}.csv', index=index)
    try:
        (ART / f'{name}.tex').write_text(tab.to_latex(index=index, escape=True, float_format='%.3f'))
    except Exception as err:                   # jinja2 absent : seul le CSV est écrit
        print('LaTeX non généré :', err)
    display(tab)

if FIG_SOURCE == 'nested' and RUN_NESTED:
    P_FIN, T_FIN = p_nest, t_nest
else:
    P_FIN, T_FIN = RUNS[(BEST_ALL, MAIN)]['p'], RUNS[(BEST_ALL, MAIN)]['t']
PRED_FIN = P_FIN.argmax(1)
print('Modèle des figures :', exp_label(BEST_ALL) if FIG_SOURCE != 'nested' else 'sélection imbriquée')

### Tableau 1 — Caractéristiques de la cohorte

In [ ]:
from scipy.stats import kruskal, chi2_contingency
def fmt_cont(x, d=1):
    s = f'{np.nanmean(x):.{d}f} ± {np.nanstd(x):.{d}f}'
    return s if EN else s.replace('.', ',')
def fmt_cat(mask, sub):
    return f'{int(mask[sub].sum())} ({mask[sub].mean() * 100:.0f}%)'
groups = [Y == c for c in range(3)]
rows = {}
for lab, col, d in [(L('Âge (ans)', 'Age (years)'), 'age', 1), (L('Taille (m)', 'Height (m)'), 'height', 2),
                    (L('Poids (kg)', 'Weight (kg)'), 'weight', 1), (L('IMC (kg/m²)', 'BMI (kg/m²)'), 'BMI', 1),
                    (L('T-score (DXA)', 'T-score (DXA)'), 't_value', 2), (L('Score OST', 'OST score'), 'OST', 1)]:
    v = df[col].values
    rows[lab] = [fmt_cont(v, d)] + [fmt_cont(v[g], d) for g in groups] + [kruskal(*[v[g] for g in groups]).pvalue]
for lab, mask in [(L('Femmes', 'Female'), df.sex_f.values == 1), (L('Scanner disponible', 'CT available'), PAT_CT >= 0),
                  (L('Deux vues radiographiques', 'Two radiographic views'), (PAT_X1 >= 0).sum(1) == 2),
                  (L('Matériel chirurgical', 'Spinal implants / cement'), HAS_HW)]:
    tab = np.array([[(mask & g).sum(), (~mask & g).sum()] for g in groups])
    rows[lab] = [fmt_cat(mask, np.ones(N_PAT, bool))] + [fmt_cat(mask, g) for g in groups] + [chi2_contingency(tab)[1]]
table1 = pd.DataFrame(rows, index=[f'{L("Total", "All")} (n = {N_PAT})'] + [f'{c} (n = {g.sum()})' for c, g in zip(CLS_ART, groups)] + ['p']).T
table1['p'] = table1['p'].map(fmt_p)
save_table(table1, 'table1_cohort' if EN else 'tableau1_cohorte')

### Figure 1a — Diagramme de flux de la cohorte

In [ ]:
n_x = int(((PAT_X1 >= 0).sum(1) > 0).sum()); n_ct = int((PAT_CT >= 0).sum()); n_both = int((((PAT_X1 >= 0).sum(1) > 0) & (PAT_CT >= 0)).sum())
n_clin_xlsx = 803
boxes = [
    (0.5, 0.92, L(f'Jeu de données LUMOS\n{n_clin_xlsx} patients', f'LUMOS dataset\n{n_clin_xlsx} patients')),
    (0.5, 0.74, L(f'Patients avec images et étiquette\nn = {N_PAT}', f'Patients with images and label\nn = {N_PAT}')),
    (0.17, 0.52, L(f'Radiographies\nn = {n_x} patients\n{len(x_stems)} images (AP / LAT)', f'Radiographs\nn = {n_x} patients\n{len(x_stems)} images (AP / LAT)')),
    (0.5, 0.52, L(f'Scanner (CT)\nn = {n_ct} patients\n(dont {n_both} avec radiographies)', f'CT\nn = {n_ct} patients\n({n_both} with radiographs)')),
    (0.83, 0.52, L(f'Matériel chirurgical\nn = {int(HAS_HW.sum())} patients', f'Spinal implants\nn = {int(HAS_HW.sum())} patients')),
    (0.5, 0.28, L('Validation croisée à 5 plis par patient\n', '5-fold patient-level cross-validation\n') + ' / '.join(str(int((df.fold == k).sum())) for k in FOLDS) + L(' patients par pli', ' patients per fold')),
    (0.5, 0.09, f'{CLS_ART[0]} {int((Y == 0).sum())}  ·  {CLS_ART[1]} {int((Y == 1).sum())}  ·  {CLS_ART[2]} {int((Y == 2).sum())}'),
]
fig, ax = plt.subplots(figsize=(COL2 * 0.8, 4.2)); ax.axis('off')
for x, y, t in boxes:
    ax.text(x, y, t, ha='center', va='center', fontsize=7.5, bbox=dict(boxstyle='round,pad=0.5', fc='#eef3f7', ec='#2f5d7c'))
ax.text(0.86, 0.83, L(f'Exclus : {n_clin_xlsx - N_PAT}\n(sans image exploitable)', f'Excluded: {n_clin_xlsx - N_PAT}\n(no usable image)'),
        ha='center', va='center', fontsize=7, bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#999', ls='--'))
for (x0, y0), (x1, y1) in [((0.5, 0.88), (0.5, 0.79)), ((0.5, 0.70), (0.17, 0.58)), ((0.5, 0.70), (0.5, 0.58)), ((0.5, 0.70), (0.83, 0.58)),
                           ((0.5, 0.46), (0.5, 0.33)), ((0.5, 0.23), (0.5, 0.13)), ((0.5, 0.83), (0.75, 0.83))]:
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle='->', color='#2f5d7c'))
save_fig(fig, 'figure1a_flowchart' if EN else 'figure1a_flux_cohorte')

### Tableau 2 — Comparaison des modèles (IC 95 %, tests corrigés par Holm)

In [ ]:
cands_t2 = {L('Score OST', 'OST score'): (oof_ost, None),
            L('Clinique seule (régression logistique)', 'Clinical variables (logistic regression)'): (oof_clin, None),
            L('Image + clinique (sonde linéaire)', 'Image + clinical (linear probe)'): (oof_probe, None)}
lab_en = lambda e: e.replace('gelé', 'frozen').replace('fine-tuné', 'fine-tuned') if EN else e
ref_k = exp_key(*REFERENCE, False)
if (ref_k, MAIN) in RUNS and ref_k != BEST_ALL:
    cands_t2[f'PRISM — {lab_en(exp_label(ref_k))}'] = (RUNS[(ref_k, MAIN)]['p'], RUNS[(ref_k, MAIN)]['t'])
cands_t2[f'PRISM — {lab_en(exp_label(BEST_ALL))} (final)'] = (RUNS[(BEST_ALL, MAIN)]['p'], RUNS[(BEST_ALL, MAIN)]['t'])
if RUN_NESTED:
    cands_t2[L('PRISM — sélection imbriquée (sans biais)', 'PRISM — nested selection (unbiased)')] = (p_nest, t_nest)
cands_t2[L('PRISM (final) déployé sans CT', 'PRISM (final) without CT')] = (RUNS[(BEST_ALL, MAIN)]['p_noct'], RUNS[(BEST_ALL, MAIN)]['t_noct'])
rows, pd_raw, pq_raw = {}, [], []
for name, (p, t) in cands_t2.items():
    m = metrics3(Y, p, T if t is not None else None, t); ci = bootstrap_ci(Y, p)
    is_clin = p is oof_clin
    pd_raw.append(np.nan if is_clin else delong_test(Y == 2, p[:, 2], oof_clin[:, 2])[2])
    pq_raw.append(np.nan if is_clin else paired_qwk_pvalue(Y, p, oof_clin))
    rows[name] = {'QWK [95% CI]': f"{m['qwk']:.3f} [{ci['qwk_ci'][0]:.3f}–{ci['qwk_ci'][1]:.3f}]",
                  'Macro F1': m['f1_macro'], L('Acc. équilibrée', 'Balanced acc.'): m['bal_acc'], 'Macro AUC': m['auc_macro'],
                  L('AUC ostéoporose [IC 95 %]', 'Osteoporosis AUC [95% CI]'): f"{m['auc_osteoporose']:.3f} [{ci['auc_osteo_ci'][0]:.3f}–{ci['auc_osteo_ci'][1]:.3f}]",
                  L('Sens. ostéoporose', 'Osteoporosis sens.'): m['sens_osteoporose'], L('Spéc. ostéoporose', 'Osteoporosis spec.'): m['spec_osteoporose'],
                  L('EAM T-score', 'T-score MAE'): m.get('t_mae', np.nan)}
table2 = pd.DataFrame(rows).T
table2[L('p DeLong vs clinique (Holm)', 'p DeLong vs clinical (Holm)')] = [fmt_p(v) for v in holm(pd_raw)]
table2[L('p QWK vs clinique (Holm)', 'p QWK vs clinical (Holm)')] = [fmt_p(v) for v in holm(pq_raw)]
save_table(table2, 'table2_model_comparison' if EN else 'tableau2_comparaison_modeles')

### Tableau 3 — Ablation des composants (modèle final)

In [ ]:
ABL_EN = {'Clinique (MLP)': 'Clinical only (MLP)', 'Radiographie seule': 'Radiograph only', 'Radiographie + clinique': 'Radiograph + clinical',
          'PRISM sans info privilégiée': 'PRISM w/o privileged DXA targets', 'PRISM sans auto-distillation': 'PRISM w/o self-distillation',
          'PRISM sans régression du T-score': 'PRISM w/o T-score regression', MAIN: 'PRISM (full)'}
if RUN_ABLATIONS:
    names = [c for c in CONFIGS if (BEST, c) in RUNS]
    full = RUNS[(BEST, MAIN)]
    t3 = {}
    praw = []
    for c in names:
        r = RUNS[(BEST, c)]; m = metrics3(Y, r['p'], T, r['t'])
        t3[ABL_EN.get(c, c) if EN else c] = {'QWK': m['qwk'], 'Macro F1': m['f1_macro'], 'Macro AUC': m['auc_macro'],
                                             L('AUC ostéoporose', 'Osteoporosis AUC'): m['auc_osteoporose'],
                                             L('Δ QWK vs complet', 'Δ QWK vs full'): m['qwk'] - metrics3(Y, full['p'])['qwk']}
        praw.append(np.nan if c == MAIN else paired_qwk_pvalue(Y, r['p'], full['p']))
    t3 = pd.DataFrame(t3).T
    t3[L('p QWK vs complet (Holm)', 'p QWK vs full (Holm)')] = [fmt_p(v) for v in holm(praw)]
    save_table(t3, 'table3_ablation' if EN else 'tableau3_ablation')

### Tableau 4 — Effet de chaque piste (moyennes marginales) et grille complète

In [ ]:
# Moyennes marginales sur la grille FIGÉE uniquement (plan factoriel équilibré). Les runs fine-tunés ne couvrent
# que 2 configurations : les inclure biaiserait les niveaux concernés (ex. flag, mask) → comparés à part, appariés.
COLS4 = ['qwk', 'auc_osteoporose', 'qwk_sans_materiel']
REN4 = {'qwk': L('QWK moyen', 'Mean QWK'), 'auc_osteoporose': L('AUC ostéoporose', 'Osteoporosis AUC'),
        'qwk_sans_materiel': L('QWK sans matériel', 'QWK w/o implants')}
frozen = grid_res[~grid_res['finetune'].astype(bool)] if 'finetune' in grid_res else grid_res
t4 = []
for fac, lab in [('backbone', L('Extracteur', 'Backbone')), ('roi', L('Région', 'Region')), ('hardware', L('Matériel', 'Implants'))]:
    if frozen[fac].nunique() > 1:
        g = frozen.groupby(fac)[COLS4].mean(); g.index = [f'{lab}: {i}' for i in g.index]; t4.append(g)
        if fac != 'backbone' and frozen['backbone'].nunique() > 1:          # détail par extracteur
            gb = frozen.groupby(['backbone', fac])[COLS4].mean()
            gb.index = [f'{lab}: {f} ({b})' for b, f in gb.index]; t4.append(gb)
if t4:
    t4 = pd.concat(t4).rename(columns=REN4)
    save_table(t4, 'table4_design_choices' if EN else 'tableau4_effet_pistes')
    print(L('Moyennes marginales sur', 'Marginal means over'), len(frozen), L('configurations figées', 'frozen configurations'))

# Fine-tuning vs figé, apparié sur la même configuration (backbone, roi, hardware)
if 'finetune' in grid_res and grid_res['finetune'].astype(bool).any():
    key = ['backbone', 'roi', 'hardware']; ft = grid_res[grid_res['finetune'].astype(bool)]
    rows = []
    for _, r in ft.iterrows():
        fz = frozen.set_index(key).loc[tuple(r[key])]
        rows.append({L('Configuration', 'Configuration'): ' | '.join(map(str, r[key])),
                     L('QWK figé', 'QWK frozen'): fz['qwk'], L('QWK fine-tuné', 'QWK fine-tuned'): r['qwk'],
                     L('AUC ostéo. figé', 'Osteo. AUC frozen'): fz['auc_osteoporose'],
                     L('AUC ostéo. fine-tuné', 'Osteo. AUC fine-tuned'): r['auc_osteoporose'],
                     'Δ QWK': r['qwk'] - fz['qwk']})
    save_table(pd.DataFrame(rows).set_index(L('Configuration', 'Configuration')),
               'table4b_finetuning' if EN else 'tableau4b_finetuning')
gs1 = grid_res.copy(); gs1['experience'] = gs1['experience'].map(lab_en)
save_table(gs1, 'tableS1_full_grid' if EN else 'tableauS1_grille_complete', index=False)

### Figure 2 — Performance diagnostique : ROC (IC 95 %), précision-rappel, matrice de confusion

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
def roc_band(y, s, n_boot=500, seed=0):
    grid = np.linspace(0, 1, 101); rs = np.random.RandomState(seed); tprs, aucs = [], []
    for _ in range(n_boot):
        i = rs.randint(0, len(y), len(y))
        if y[i].min() == y[i].max():
            continue
        fpr, tpr, _ = roc_curve(y[i], s[i]); tprs.append(np.interp(grid, fpr, tpr)); aucs.append(roc_auc_score(y[i], s[i]))
    return grid, np.percentile(tprs, 2.5, 0), np.percentile(tprs, 97.5, 0), np.percentile(aucs, [2.5, 97.5])

def interp_pr(y, s):
    pr, rc, _ = precision_recall_curve(y, s)
    return np.maximum.accumulate(pr), rc                   # précision interpolée : max des précisions à rappel ≥ r

fig, ax = plt.subplots(1, 3, figsize=(COL2, 3.0), gridspec_kw=dict(width_ratios=[1, 1, 1.05]))
for c in range(3):
    yc = (Y == c).astype(int)
    fpr, tpr, _ = roc_curve(yc, P_FIN[:, c]); g, lo, hi, aci = roc_band(yc, P_FIN[:, c])
    ax[0].fill_between(g, lo, hi, color=CLS_COLORS[c], alpha=0.15, lw=0)
    ax[0].plot(fpr, tpr, color=CLS_COLORS[c], lw=1.4, label=f'{CLS_ART[c]} {roc_auc_score(yc, P_FIN[:, c]):.3f} [{aci[0]:.3f}–{aci[1]:.3f}]')
ax[0].plot([0, 1], [0, 1], 'k--', lw=0.6); ax[0].set_xlabel(L('1 − spécificité', '1 − specificity')); ax[0].set_ylabel(L('Sensibilité', 'Sensitivity'))
ax[0].set_title(L('(a) ROC one-vs-rest', '(a) One-vs-rest ROC'))
ax[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), title='AUC [95% CI]', title_fontsize=6.5, fontsize=6, frameon=False)
for p, lab, col, ls in [(P_FIN[:, 2], 'PRISM', CLS_COLORS[2], '-'), (oof_clin[:, 2], L('Clinique', 'Clinical'), GREY, '-'), (oof_ost[:, 2], 'OST', GREY, ':')]:
    pr, rc = interp_pr(Y == 2, p)
    ax[1].plot(rc, pr, lw=1.4, color=col, ls=ls, label=f'{lab} (AP {average_precision_score(Y == 2, p):.3f})')
ax[1].axhline((Y == 2).mean(), color='k', ls='--', lw=0.6, label=L(f'Prévalence ({(Y == 2).mean():.2f})', f'Prevalence ({(Y == 2).mean():.2f})'))
ax[1].set_xlabel(L('Rappel (sensibilité)', 'Recall (sensitivity)')); ax[1].set_ylabel(L('Précision (VPP)', 'Precision (PPV)'))
ax[1].set_title(L('(b) Précision-rappel — ostéoporose', '(b) Precision-recall, osteoporosis'))
ax[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.24), fontsize=6, frameon=False)
ax[1].set_ylim(0.2, 1.02)
cmn = confusion_matrix(Y, PRED_FIN, normalize='true'); cm = confusion_matrix(Y, PRED_FIN)
pct = lambda v: f'{v * 100:.0f}%' if EN else f'{v * 100:.0f} %'
annot = np.array([[f'{pct(cmn[i, j])}\n({cm[i, j]})' for j in range(3)] for i in range(3)])
sns.heatmap(cmn, annot=annot, fmt='', cmap='Blues', vmin=0, vmax=1, cbar=False, xticklabels=CLS_ART, yticklabels=CLS_ART, ax=ax[2],
            annot_kws=dict(fontsize=7))
ax[2].set_yticklabels(CLS_ART, rotation=0); ax[2].set_xticklabels(CLS_ART, rotation=25, ha='right')
ax[2].set_xlabel(L('Prédit', 'Predicted')); ax[2].set_ylabel(L('Réel (DXA)', 'Reference (DXA)'))
ax[2].set_title(L('(c) Matrice de confusion', '(c) Confusion matrix (row %)'))
plt.tight_layout(); save_fig(fig, 'figure2_performance')

### Figure 3 — T-score prédit : corrélation et Bland-Altman

In [ ]:
from scipy.stats import pearsonr
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.7))
ax[0].scatter(T, T_FIN, c=[CLS_COLORS[c] for c in Y], s=5, alpha=0.6, lw=0)
lim = [min(T.min(), T_FIN.min()) - .3, max(T.max(), T_FIN.max()) + .3]
ax[0].plot(lim, lim, 'k--', lw=0.6)
for v in (-1, -2.5):
    ax[0].axvline(v, color='gray', ls=':', lw=0.6); ax[0].axhline(v, color='gray', ls=':', lw=0.6)
r, _ = pearsonr(T, T_FIN)
ax[0].set_xlabel('DXA T-score'); ax[0].set_ylabel(L('T-score prédit', 'Predicted T-score'))
ax[0].set_title(f'(a) r = {r:.2f}, ρ = {spearmanr(T, T_FIN).correlation:.2f}, MAE = {np.mean(np.abs(T - T_FIN)):.2f}')
mean_, diff = (T + T_FIN) / 2, T_FIN - T
bias, sd = diff.mean(), diff.std(ddof=1)
ax[1].scatter(mean_, diff, c=[CLS_COLORS[c] for c in Y], s=5, alpha=0.6, lw=0)
for v, ls, lab in [(bias, '-', f'{L("Biais", "Bias")} {bias:+.2f}'), (bias + 1.96 * sd, '--', f'+1.96 SD {bias + 1.96 * sd:+.2f}'),
                   (bias - 1.96 * sd, '--', f'−1.96 SD {bias - 1.96 * sd:+.2f}')]:
    ax[1].axhline(v, color='k', ls=ls, lw=0.7); ax[1].text(ax[1].get_xlim()[1], v, ' ' + lab, va='center', fontsize=6.5)
ax[1].set_xlabel(L('Moyenne (DXA, prédit)', 'Mean of DXA and predicted')); ax[1].set_ylabel(L('Prédit − DXA', 'Predicted − DXA'))
ax[1].set_title('(b) Bland–Altman')
handles = [plt.Line2D([], [], marker='o', ls='', ms=4, color=CLS_COLORS[c], label=CLS_ART[c]) for c in range(3)]
ax[0].legend(handles=handles, loc='upper left')
plt.tight_layout(); save_fig(fig, 'figure3_tscore')

### Figure 4 — Cartes d'attention : succès et erreurs

In [ ]:
set_experiment(*BEST[:3])
p_best = RUNS[(BEST, MAIN)]['p']; pred_best = p_best.argmax(1)
rng = np.random.RandomState(1); picks = []
for c in range(3):
    ok = np.where((Y == c) & (pred_best == c) & (PAT_X1[:, 0] >= 0))[0]
    picks.append((L('Correct', 'Correct'), rng.choice(ok)))
err_grave = np.where((np.abs(Y - pred_best) == 2) & (PAT_X1[:, 0] >= 0))[0]
err_voisin = np.where((np.abs(Y - pred_best) == 1) & (PAT_X1[:, 0] >= 0))[0]
if len(err_voisin): picks.append((L('Erreur voisine', 'Adjacent error'), rng.choice(err_voisin)))
if len(err_grave): picks.append((L('Erreur grave', 'Severe error'), rng.choice(err_grave)))
parts_b = ['full', 'spine'] if BEST[1] == 'full+spine' else [BEST[1]]
fig, axes = plt.subplots(1, len(picks), figsize=(COL2, 2.9))
for ax, (kind, i) in zip(np.atleast_1d(axes), picks):
    att = attention_maps(i)
    v = 0 if PAT_X[i, 0] >= 0 else 1
    kpart, s = divmod(int(PAT_X[i, v]), N_img)
    img = (load_xray if parts_b[kpart] == 'full' else load_roi)(x_stems[s])
    ax.imshow(img, cmap='gray')
    ax.imshow(np.asarray(Image.fromarray((att[v] / att[v].max() * 255).astype(np.uint8)).resize(img.shape[::-1], Image.BICUBIC)), cmap='jet', alpha=0.35)
    ax.axis('off')
    ax.set_title(f'{kind}\nDXA: {CLS_ART[Y[i]]} (T = {T[i]:.1f})\n{L("Prédit", "Predicted")}: {CLS_ART[pred_best[i]]}', fontsize=6.5,
                 color='#b8322a' if kind != 'Correct' else 'black')
plt.tight_layout(); save_fig(fig, 'figure4_attention')

### Figure 5 — Comparaison des configurations (QWK, IC 95 %)

In [ ]:
g = grid_res.iloc[::-1]
ci_ = g.qwk_ci.str.extract(r'\[([-\d.]+)–([-\d.]+)\]').astype(float)
fig, ax = plt.subplots(figsize=(COL1 * 1.25, 0.16 * len(g) + 0.9))
cols = ['#b8322a' if ft else '#2f5d7c' for ft in g.finetune]
ax.errorbar(g.qwk, range(len(g)), xerr=[g.qwk - ci_[0], ci_[1] - g.qwk], fmt='none', ecolor='#9aa5b1', capsize=1.5, lw=0.7)
ax.scatter(g.qwk, range(len(g)), c=cols, zorder=3, s=8)
ax.axvline(RESULTS['Référence clinique (LR)']['qwk'], color=GREY, ls='--', lw=0.8, label=L('Clinique seule', 'Clinical only'))
if RUN_NESTED:
    ax.axvline(m_nest['qwk'], color='#2f7d5b', ls='-', lw=0.9, label=L('Sélection imbriquée', 'Nested selection'))
ax.set_yticks(range(len(g))); ax.set_yticklabels([lab_en(e) for e in g.experience], fontsize=5.5); ax.set_xlabel('QWK (95% CI)')
ax.legend(loc='lower right', fontsize=6); plt.tight_layout(); save_fig(fig, 'figure5_grid')

### Figure 6 — Calibration et analyse de la courbe de décision (DCA)

In [ ]:
from sklearn.calibration import calibration_curve
def net_benefit(y, p, thr):
    pred = p >= thr; n = len(y)
    tp = (pred & (y == 1)).sum(); fp = (pred & (y == 0)).sum()
    return tp / n - fp / n * thr / (1 - thr)
y2 = (Y == 2).astype(int)
fig, ax = plt.subplots(1, 2, figsize=(COL2, 2.7))
for p, lab, col, ls in [(P_FIN[:, 2], 'PRISM', CLS_COLORS[2], '-'), (oof_clin[:, 2], L('Clinique seule', 'Clinical only'), GREY, '-')]:
    fr, mp = calibration_curve(y2, p, n_bins=10, strategy='quantile')
    ax[0].plot(mp, fr, 'o', ls=ls, color=col, lw=1.2, ms=3, label=f'{lab} (ECE ≈ {np.mean(np.abs(fr - mp)):.3f})')
ax[0].plot([0, 1], [0, 1], 'k--', lw=0.6); ax[0].set_xlabel(L('Probabilité prédite (ostéoporose)', 'Predicted probability (osteoporosis)'))
ax[0].set_ylabel(L('Fréquence observée', 'Observed frequency')); ax[0].set_title('(a) Calibration'); ax[0].legend()
thr = np.linspace(0.05, 0.6, 56)
ax[1].plot(thr, [net_benefit(y2, P_FIN[:, 2], t) for t in thr], color=CLS_COLORS[2], lw=1.4, label='PRISM')
ax[1].plot(thr, [net_benefit(y2, oof_clin[:, 2], t) for t in thr], color=GREY, lw=1.2, label=L('Clinique seule', 'Clinical only'))
ax[1].plot(thr, [net_benefit(y2, oof_ost[:, 2], t) for t in thr], color=GREY, lw=1.2, ls=':', label='OST')
ax[1].plot(thr, [net_benefit(y2, np.ones(N_PAT), t) for t in thr], color='k', lw=0.8, ls='--', label=L('Adresser tout le monde', 'Refer all'))
ax[1].axhline(0, color='k', lw=0.6, label=L('Personne', 'Refer none'))
ax[1].set_ylim(-0.05, y2.mean() + 0.03); ax[1].set_xlabel(L('Seuil de probabilité', 'Threshold probability')); ax[1].set_ylabel(L('Bénéfice net', 'Net benefit'))
ax[1].set_title(L('(b) Courbe de décision', '(b) Decision curve')); ax[1].legend()
plt.tight_layout(); save_fig(fig, 'figure6_calibration_dca')

### Tableau S2 — Performance par sous-groupe

In [ ]:
age_t = pd.cut(df.age, [0, 55, 70, 200], labels=['< 55', '55–70', '> 70'])
subsets = {L('Tous', 'All'): np.ones(N_PAT, bool), L('Femmes', 'Female'): df.sex_f.values == 1, L('Hommes', 'Male'): df.sex_f.values == 0,
           **{f'{L("Âge", "Age")} {a}': (age_t == a).values for a in age_t.cat.categories},
           L('Avec matériel', 'With implants'): HAS_HW, L('Sans matériel', 'Without implants'): ~HAS_HW,
           L('Avec CT', 'With CT'): PAT_CT >= 0, L('Sans CT', 'Without CT'): PAT_CT < 0,
           L('Deux vues', 'Two views'): (PAT_X1 >= 0).sum(1) == 2, L('Une vue', 'One view'): (PAT_X1 >= 0).sum(1) == 1}
rows = {}
for name, msk in subsets.items():
    if msk.sum() < 20 or len(np.unique(Y[msk])) < 3:
        continue
    m = metrics3(Y[msk], P_FIN[msk])
    rows[name] = {'n': int(msk.sum()), 'QWK': m['qwk'], 'Macro F1': m['f1_macro'],
                  L('AUC ostéoporose', 'Osteoporosis AUC'): m['auc_osteoporose'], L('Sens. ostéoporose', 'Osteoporosis sens.'): m['sens_osteoporose']}
save_table(pd.DataFrame(rows).T, 'tableS2_subgroups' if EN else 'tableauS2_sous_groupes')

### Tableau S3 — Erreurs selon la distance au seuil diagnostique

In [ ]:
dist = np.minimum(np.abs(T + 1), np.abs(T + 2.5))
bins = pd.cut(dist, [0, 0.25, 0.5, 1.0, 10], labels=['< 0.25', '0.25–0.5', '0.5–1', '≥ 1'], include_lowest=True)
t_s3 = pd.DataFrame({'bin': bins, 'ok': PRED_FIN == Y}).groupby('bin', observed=False).agg(n=('ok', 'size'), accuracy=('ok', 'mean'))
t_s3[L('part des erreurs', 'share of errors')] = pd.DataFrame({'bin': bins, 'err': PRED_FIN != Y}).groupby('bin', observed=False).err.sum() / (PRED_FIN != Y).sum()
t_s3.index.name = L('Distance du T-score au seuil le plus proche', 'Distance of T-score to nearest threshold')
save_table(t_s3, 'tableS3_errors_vs_threshold' if EN else 'tableauS3_erreurs_seuil')

### Tableau S4 — Points de fonctionnement pour le dépistage de l'ostéoporose

In [ ]:
rows = {}
for target in (0.80, 0.85, 0.90, 0.95):
    fpr, tpr, thr_ = roc_curve(Y == 2, P_FIN[:, 2])
    i = int(np.argmax(tpr >= target)); t_ = thr_[i]
    pred = P_FIN[:, 2] >= t_
    tp, fp = (pred & (Y == 2)).sum(), (pred & (Y != 2)).sum()
    rows[f'{L("Sensibilité", "Sensitivity")} ≥ {target:.2f}'] = {L('Seuil', 'Threshold'): t_, L('Sensibilité', 'Sensitivity'): tpr[i],
        L('Spécificité', 'Specificity'): 1 - fpr[i], 'PPV': tp / max(pred.sum(), 1), 'NPV': ((~pred) & (Y != 2)).sum() / max((~pred).sum(), 1),
        L('Patients adressés en DXA', 'Patients referred to DXA'): int(pred.sum())}
save_table(pd.DataFrame(rows).T, 'tableS4_operating_points' if EN else 'tableauS4_points_fonctionnement')
print('Fichiers exportés dans', ART)

## 17. Benchmark des extracteurs d'images
Même protocole pour tous : **tête PRISM complète, image entière, matériel non traité, extracteur gelé, 5 plis × `GRID_SEEDS`**. Seul l'extracteur change.

| Extracteur | Pourquoi |
|---|---|
| ResNet-50, DenseNet-121, EfficientNet-B3, ConvNeXt-Base | CNN pré-entraînés sur ImageNet, les plus utilisés dans la littérature |
| **DINOv2 ViT-B/14** | **Contrôle clé** : même architecture et même méthode que RAD-DINO, mais entraîné sur des photos. L'écart DINOv2 → RAD-DINO mesure l'apport du pré-entraînement sur des **radiographies**. |
| RAD-DINO ViT-B/14 | Modèle de fondation radiologique |

Les expériences déjà calculées (ResNet-50, RAD-DINO) sont rechargées. Durée indicative sur Mac : 30 min à 1 h par nouvel extracteur.

In [ ]:
RUN_BENCH = True
BENCH_BACKBONES = ['resnet50', 'densenet121', 'efficientnet_b3', 'convnext_base', 'dinov2', 'raddino']
BENCH_ROI, BENCH_HW = 'full', 'keep'
if os.environ.get('LUMOS_BENCH'):
    BENCH_BACKBONES = json.loads(os.environ['LUMOS_BENCH'])

def backbone_params(bb):
    f = CACHE / 'backbone_params.json'
    d = json.loads(f.read_text()) if f.exists() else {}
    if bb not in d:
        net = Backbone(bb)
        d[bb] = sum(p.numel() for p in net.parameters()) / 1e6
        f.write_text(json.dumps(d)); del net; free_gpu_memory()
    return d[bb]

if RUN_BENCH:
    bench_runs = {}
    for bb in BENCH_BACKBONES:
        r = run_experiment(exp_key(bb, BENCH_ROI, BENCH_HW, False))
        if r is not None:
            bench_runs[bb] = r
    ref_bb = 'raddino' if 'raddino' in bench_runs else list(bench_runs)[-1]
    rows = {}
    for bb, r in bench_runs.items():
        m = metrics3(Y, r['p'], T, r['t']); ci = bootstrap_ci(Y, r['p'])
        row = {'Pré-entraînement': BACKBONES[bb]['pretrain'], 'Paramètres (M)': round(backbone_params(bb), 1),
               'QWK': m['qwk'], 'QWK [IC 95 %]': f"{m['qwk']:.3f} [{ci['qwk_ci'][0]:.3f}–{ci['qwk_ci'][1]:.3f}]",
               'F1 macro': m['f1_macro'], 'AUC macro': m['auc_macro'],
               'AUC ostéoporose [IC 95 %]': f"{m['auc_osteoporose']:.3f} [{ci['auc_osteo_ci'][0]:.3f}–{ci['auc_osteo_ci'][1]:.3f}]",
               'Sens. ostéoporose': m['sens_osteoporose'], 'EAM T-score': m['t_mae']}
        row['_pd'] = np.nan if bb == ref_bb else delong_test(Y == 2, r['p'][:, 2], bench_runs[ref_bb]['p'][:, 2])[2]
        row['_pq'] = np.nan if bb == ref_bb else paired_qwk_pvalue(Y, r['p'], bench_runs[ref_bb]['p'])
        rows[bb] = row
    bench = pd.DataFrame(rows).T
    bench[f'p DeLong vs {ref_bb} (Holm)'] = [fmt_p(v) for v in holm(bench['_pd'].astype(float))]
    bench[f'p QWK vs {ref_bb} (Holm)'] = [fmt_p(v) for v in holm(bench['_pq'].astype(float))]
    bench = bench.drop(columns=['_pd', '_pq']).sort_values('QWK')
    if EN:
        PRE_EN = {'ImageNet-1k, supervisé': 'ImageNet-1k, supervised', 'ImageNet-22k, supervisé': 'ImageNet-22k, supervised',
                  'photos naturelles (LVD-142M), auto-supervisé': 'natural images (LVD-142M), self-supervised',
                  'radiographies, auto-supervisé': 'radiographs, self-supervised'}
        bench['Pré-entraînement'] = bench['Pré-entraînement'].map(lambda v: PRE_EN.get(v, v))
        bench = bench.rename(columns={'Pré-entraînement': 'Pre-training', 'Paramètres (M)': 'Params (M)', 'QWK [IC 95 %]': 'QWK [95% CI]',
                                      'F1 macro': 'Macro F1', 'AUC macro': 'Macro AUC', 'AUC ostéoporose [IC 95 %]': 'Osteoporosis AUC [95% CI]',
                                      'Sens. ostéoporose': 'Osteoporosis sens.', 'EAM T-score': 'T-score MAE'})
    ART.mkdir(exist_ok=True)
    save_table(bench.drop(columns='QWK'), 'table5_backbone_benchmark' if EN else 'tableau5_benchmark_extracteurs')

    # Figure : QWK et AUC ostéoporose par extracteur (IC 95 %)
    fig, ax = plt.subplots(1, 2, figsize=(COL2, 0.28 * len(bench) + 1.0))
    names = list(bench.index)
    cols_bb = ['#2f7d5b' if b == 'raddino' else ('#d08a1e' if b == 'dinov2' else '#2f5d7c') for b in names]
    qcol, acol = ('QWK [95% CI]', 'Osteoporosis AUC [95% CI]') if EN else ('QWK [IC 95 %]', 'AUC ostéoporose [IC 95 %]')
    for a, metric, lab in [(ax[0], qcol, 'QWK'), (ax[1], acol, L('AUC ostéoporose', 'Osteoporosis AUC'))]:
        vals = bench[metric].str.extract(r'([\d.]+) \[([\d.]+)–([\d.]+)\]').astype(float)
        a.errorbar(vals[0], range(len(names)), xerr=[vals[0] - vals[1], vals[2] - vals[0]], fmt='none', ecolor='#9aa5b1', capsize=3)
        a.scatter(vals[0], range(len(names)), c=cols_bb, s=40, zorder=3)
        a.set_yticks(range(len(names))); a.set_yticklabels(names); a.set_xlabel(f'{lab} ' + L('(IC 95 %)', '(95% CI)'))
    ax[0].axvline(RESULTS['Référence clinique (LR)']['qwk'], color='#7a8794', ls='--', lw=0.8, label=L('Clinique seule', 'Clinical only')); ax[0].legend(loc='lower right')
    ax[0].set_title(L('(a) Classification à 3 classes', '(a) Three-class classification')); ax[1].set_title(L("(b) Dépistage de l'ostéoporose", '(b) Osteoporosis screening'))
    plt.tight_layout(); save_fig(fig, 'figure7_backbone_benchmark' if EN else 'figure7_benchmark_extracteurs')

## 18. Ablation de l'encodeur clinique : le MLP est-il nécessaire ?
Question : l'encodeur clinique **MLP (5 → 256 → 256)** et la modulation **FiLM** apportent-ils quelque chose par rapport à des choix plus simples ?

Toutes les variantes reprennent le **modèle final** (`FINAL_MODEL`) avec la tête PRISM complète, les **mêmes plis et les mêmes graines**. Seul le traitement des 5 variables cliniques change.

| Variante | Encodeur clinique | FiLM | Clinique dans la fusion |
|---|---|---|---|
| A — PRISM (actuel) | MLP 5 → 256 → 256 | oui | oui |
| B | linéaire 5 → 256 | oui | oui |
| C | MLP | non | oui |
| D | linéaire | non | oui |
| E — sans clinique | — | non | non |
| F — fusion tardive | régression logistique sur [log-probabilités de E + 5 variables cliniques] | — | — |

A, B, C et D forment un **plan 2 × 2** (MLP ou linéaire × FiLM ou non) : on sépare l'effet de l'encodeur de celui de FiLM.

**Règle de décision** (la même que pour le modèle final) : une variante plus simple est jugée équivalente à A si sa perte de QWK est inférieure à 0,01 **et** non significative après correction de Holm (p > 0,05). On retient alors la plus simple.

Un contrôle « clinique seule » compare aussi, **à protocole identique**, le MLP et l'encodeur linéaire, ainsi que la régression logistique de référence.

Durée : 5 nouveaux entraînements (chacun 5 plis × `GRID_SEEDS`), comparable à 5 configurations de la grille. Les résultats sont enregistrés et rechargés comme les autres expériences.

In [ ]:
RUN_CLIN_ABLATION = True
K_CLIN = exp_key(*FINAL_MODEL, False) if FINAL_MODEL else BEST
FULL_MODS = {'xray', 'ct', 'clin'}
CLIN_VARIANTS = {
    'A — MLP + FiLM (PRISM)':        MAIN,
    'B — linéaire + FiLM':           'Clinique : linéaire + FiLM',
    'C — MLP sans FiLM':             'Clinique : MLP sans FiLM',
    'D — linéaire sans FiLM':        'Clinique : linéaire sans FiLM',
    'E — sans variables cliniques':  'Sans variables cliniques',
}
CONFIGS.update({
    'Clinique : linéaire + FiLM':    dict(mods=FULL_MODS, priv=True, distill=True, clin_enc='linear', film=True),
    'Clinique : MLP sans FiLM':      dict(mods=FULL_MODS, priv=True, distill=True, clin_enc='mlp', film=False),
    'Clinique : linéaire sans FiLM': dict(mods=FULL_MODS, priv=True, distill=True, clin_enc='linear', film=False),
    'Sans variables cliniques':      dict(mods={'xray', 'ct'}, priv=True, distill=True, film=False),
    'Clinique seule (linéaire)':     dict(mods={'clin'}, priv=False, distill=False, clin_enc='linear', film=False),
})

def clin_params(cfg):
    """Nombre de paramètres de la partie clinique (encodeur + FiLM) pour une configuration."""
    if 'clin' not in cfg['mods']:
        return 0
    m = PRISM(D_TOK, N_TOK, CT_SLICES, len(PRIV), len(CLIN_USED), d_ct=D_CT,
              clin_enc=cfg.get('clin_enc', 'mlp'), film=cfg.get('film', True))
    n = sum(p.numel() for p in m.clin_mlp.parameters())
    return n + (sum(p.numel() for p in m.film.parameters()) if m.film is not None else 0)

if RUN_CLIN_ABLATION:
    clin_runs = {}
    for lab, cname in CLIN_VARIANTS.items():
        r = run_experiment(K_CLIN, cname)
        if r is not None:
            clin_runs[lab] = r
    set_experiment(*K_CLIN[:3])                       # globals (D_TOK, CLIN_USED…) du modèle final

    # F — fusion tardive : régression logistique par pli sur [log-probabilités de E, variables cliniques standardisées].
    # Remarque : les probabilités hors pli de E utilisées pour l'entraînement de F proviennent de modèles qui ont vu
    # le pli test ; F est donc légèrement optimiste. Si F n'égale pas A malgré ce biais, la conclusion est robuste.
    rE = clin_runs['E — sans variables cliniques']
    feat = np.hstack([np.log(np.clip(rE['p'], 1e-6, 1)), df[CLIN].values.astype(float)])
    p_late = np.zeros((N_PAT, 3)); coefs = []
    for k in FOLDS:
        tr, te = (df.fold != k).values, (df.fold == k).values
        sc = StandardScaler().fit(feat[tr])
        lr = LogisticRegression(max_iter=5000).fit(sc.transform(feat[tr]), Y[tr])
        p_late[te] = lr.predict_proba(sc.transform(feat[te])); coefs.append(lr.coef_)
    clin_runs['F — fusion tardive (LR)'] = dict(p=p_late, t=None)

    ref = clin_runs['A — MLP + FiLM (PRISM)']
    q_ref = metrics3(Y, ref['p'])['qwk']
    rows, pq, pd_ = {}, [], []
    for lab, r in clin_runs.items():
        m = metrics3(Y, r['p'], T, r['t']) if r.get('t') is not None else metrics3(Y, r['p'])
        ci = bootstrap_ci(Y, r['p'])
        cfg = CONFIGS[CLIN_VARIANTS[lab]] if lab in CLIN_VARIANTS else None
        rows[lab] = {L('Paramètres cliniques', 'Clinical params'): clin_params(cfg) if cfg else (3 + len(CLIN) + 1) * 3,
                     L('QWK [IC 95 %]', 'QWK [95% CI]'): f"{m['qwk']:.3f} [{ci['qwk_ci'][0]:.3f}–{ci['qwk_ci'][1]:.3f}]",
                     'Macro AUC': m['auc_macro'],
                     L('AUC ostéoporose', 'Osteoporosis AUC'): m['auc_osteoporose'],
                     L('EAM T-score', 'T-score MAE'): m.get('t_mae', np.nan),
                     L('Δ QWK vs A', 'Δ QWK vs A'): m['qwk'] - q_ref}
        is_ref = r is ref
        pq.append(np.nan if is_ref else paired_qwk_pvalue(Y, r['p'], ref['p']))
        pd_.append(np.nan if is_ref else delong_test(Y == 2, r['p'][:, 2], ref['p'][:, 2])[2])
    t_clin = pd.DataFrame(rows).T
    t_clin[L('p QWK vs A (Holm)', 'p QWK vs A (Holm)')] = [fmt_p(v) for v in holm(pq)]
    t_clin[L('p DeLong vs A (Holm)', 'p DeLong vs A (Holm)')] = [fmt_p(v) for v in holm(pd_)]
    if EN:
        t_clin.index = [i.replace('linéaire', 'linear').replace('sans FiLM', 'w/o FiLM').replace('sans variables cliniques', 'no clinical variables')
                         .replace('fusion tardive', 'late fusion') for i in t_clin.index]
    save_table(t_clin, 'tableS5_clinical_encoder' if EN else 'tableauS5_encodeur_clinique')

    # Coefficients de la fusion tardive (moyenne sur les plis, classe « ostéoporose » contre référence) : lisibles par un clinicien
    names = ['log p(normal)', 'log p(osteopenia)', 'log p(osteoporosis)'] + CLIN
    coef = pd.DataFrame(np.mean(coefs, 0).T, index=names, columns=CLS_ART).round(3)
    print(L('Coefficients standardisés de la fusion tardive (moyenne des 5 plis) :', 'Standardized late-fusion coefficients (mean over 5 folds):'))
    display(coef)

    # Contrôle « clinique seule » à protocole identique : LR de référence, MLP, linéaire
    r_mlp = run_experiment(K_CLIN, 'Clinique (MLP)'); r_lin = run_experiment(K_CLIN, 'Clinique seule (linéaire)')
    set_experiment(*K_CLIN[:3])
    ctrl = {L('Régression logistique (référence)', 'Logistic regression (baseline)'): oof_clin}
    if r_mlp is not None: ctrl[L('Tête PRISM, MLP', 'PRISM head, MLP')] = r_mlp['p']
    if r_lin is not None: ctrl[L('Tête PRISM, linéaire', 'PRISM head, linear')] = r_lin['p']
    base = ctrl[list(ctrl)[0]]
    t_ctrl = pd.DataFrame({k: {'QWK': metrics3(Y, p)['qwk'], L('AUC ostéoporose', 'Osteoporosis AUC'): metrics3(Y, p)['auc_osteoporose'],
                               L('p QWK vs LR', 'p QWK vs LR'): (np.nan if p is base else paired_qwk_pvalue(Y, p, base))}
                           for k, p in ctrl.items()}).T
    t_ctrl[L('p QWK vs LR', 'p QWK vs LR')] = [fmt_p(v) for v in holm(t_ctrl[L('p QWK vs LR', 'p QWK vs LR')].astype(float).values)]
    save_table(t_ctrl, 'tableS6_clinical_only' if EN else 'tableauS6_clinique_seule')

    # Verdict selon la règle de décision : équivalente = perte de QWK < 0,01 et p (Holm) > 0,05
    p_holm = dict(zip(clin_runs, holm(pq)))
    equiv = []
    for lab in ['E — sans variables cliniques', 'D — linéaire sans FiLM', 'B — linéaire + FiLM', 'C — MLP sans FiLM', 'F — fusion tardive (LR)']:
        d = metrics3(Y, clin_runs[lab]['p'])['qwk'] - q_ref
        ok = bool(d > -0.01 and p_holm[lab] > 0.05)
        if ok:
            equiv.append(lab)
        print(f"{'✔ équivalente' if ok else '✖ inférieure '} | {lab:32s} Δ QWK = {d:+.3f}   p (Holm) = {p_holm[lab]:.3f}")
    print('→ Variante la plus simple équivalente à A :', equiv[0] if equiv else 'aucune (le MLP + FiLM est justifié)')

## 19. Encodeur des coupes CT : comparaison RAD-DINO, ResNet-50 et DINOv2
Le modèle principal encode les coupes CT avec `CT_BACKBONE` (par défaut **RAD-DINO**, le même encodeur que les radiographies, par cohérence). Cette section montre que ce choix ne change pas les résultats par rapport à ResNet-50 (ImageNet) et DINOv2.

| Encodeur CT | Pré-entraînement | Rôle |
|---|---|---|
| ResNet-50 | ImageNet, supervisé | CNN supervisé classique |
| RAD-DINO (modèle principal) | radiographies thoraciques, auto-supervisé | même encodeur que la branche radiographique |
| DINOv2 | images naturelles, auto-supervisé | contrôle : effet du domaine ou de la méthode ? |

**Protocole** : modèle final (`FINAL_MODEL`), tête PRISM complète, mêmes plis et mêmes graines ; **seules les caractéristiques des coupes CT changent** (mêmes recadrages 16 × 128 px, ACP avec blanchiment → 384, sans labels). Pour les ViT, chaque coupe est agrandie à 518 × 518 et décrite par la moyenne de ses tokens de patch (l'équivalent du pooling moyen global du ResNet).

**Trois niveaux d'analyse :**
1. toute la cohorte (n = 796) ;
2. **le sous-groupe avec CT** (n ≈ 256) : c'est là que l'effet doit se voir ;
3. une **sonde linéaire** indépendante de PRISM : moyenne des tokens CT → T-score (ridge) et → ostéoporose (régression logistique), en validation croisée sur les mêmes plis. Elle mesure la qualité intrinsèque de la représentation CT.

Les p-valeurs comparent chaque encodeur à l'encodeur du modèle principal (`CT_BACKBONE`), avec correction de Holm.

Durée : extraction CT ≈ 5 min par ViT, puis 2 entraînements (5 plis × `GRID_SEEDS`).

In [ ]:
RUN_CT_ENCODER = True
CT_BACKBONES = ['resnet50', 'raddino', 'dinov2']
K_CT = exp_key(*FINAL_MODEL, False) if FINAL_MODEL else BEST
def ct_tokens(feats):
    """Même ACP avec blanchiment que la branche CT d'origine (ajustée sur toutes les coupes, sans labels)."""
    mu, W = fit_token_pca(feats.reshape(-1, feats.shape[-1]), PCA_DIM or feats.shape[-1])
    return ((feats.float() - mu) @ W).half()

if RUN_CT_ENCODER:
    CTT_REF, D_CT_REF, CT_REF = CTT, D_CT, CT_ACTIVE      # tokens CT du modèle principal, restaurés à la fin
    main_run = run_experiment(K_CT, MAIN)
    ct_feats, ct_runs = {}, {}
    try:
        for bb in CT_BACKBONES:
            try:
                ct_feats[bb] = extract_ct_features(bb)
            except Exception as err:                # ex. modèle non téléchargeable
                print('✖', bb, ': ignoré ->', repr(err)[:200]); continue
            if bb == CT_REF:
                ct_runs[bb] = main_run
            else:                                   # même configuration, seuls les tokens CT changent
                CTT = ct_tokens(ct_feats[bb]); D_CT = CTT.shape[-1]; CT_ACTIVE = bb
                ct_runs[bb] = run_experiment(K_CT, MAIN)
    finally:
        CTT, D_CT, CT_ACTIVE = CTT_REF, D_CT_REF, CT_REF
        RUNS[(K_CT, MAIN)] = main_run               # les autres sections gardent le modèle principal
    ct_runs = {k: v for k, v in ct_runs.items() if v is not None}

    HAS_CT = PAT_CT >= 0
    print(f'Sous-groupe avec CT : {HAS_CT.sum()} patients')
    ref = ct_runs[CT_REF]
    rows, p_all, p_ct, pd_all, pd_ct = {}, [], [], [], []
    for bb, r in ct_runs.items():
        m = metrics3(Y, r['p'], T, r['t']); mc = metrics3(Y[HAS_CT], r['p'][HAS_CT], T[HAS_CT], r['t'][HAS_CT])
        ci = bootstrap_ci(Y, r['p']); cic = bootstrap_ci(Y[HAS_CT], r['p'][HAS_CT])
        rows[bb] = {L('Pré-entraînement', 'Pre-training'): BACKBONES[bb]['pretrain'],
                    L('QWK (tous) [IC 95 %]', 'QWK (all) [95% CI]'): f"{m['qwk']:.3f} [{ci['qwk_ci'][0]:.3f}–{ci['qwk_ci'][1]:.3f}]",
                    L('AUC ostéo. (tous)', 'Osteo. AUC (all)'): m['auc_osteoporose'],
                    L('QWK (CT) [IC 95 %]', 'QWK (CT) [95% CI]'): f"{mc['qwk']:.3f} [{cic['qwk_ci'][0]:.3f}–{cic['qwk_ci'][1]:.3f}]",
                    L('AUC ostéo. (CT)', 'Osteo. AUC (CT)'): mc['auc_osteoporose'],
                    L('EAM T-score (CT)', 'T-score MAE (CT)'): mc['t_mae'],
                    L(f'Δ QWK (CT) vs {CT_REF}', f'Δ QWK (CT) vs {CT_REF}'): mc['qwk'] - metrics3(Y[HAS_CT], ref['p'][HAS_CT])['qwk']}
        same = r is ref
        p_all.append(np.nan if same else paired_qwk_pvalue(Y, r['p'], ref['p']))
        p_ct.append(np.nan if same else paired_qwk_pvalue(Y[HAS_CT], r['p'][HAS_CT], ref['p'][HAS_CT]))
        pd_all.append(np.nan if same else delong_test(Y == 2, r['p'][:, 2], ref['p'][:, 2])[2])
        pd_ct.append(np.nan if same else delong_test(Y[HAS_CT] == 2, r['p'][HAS_CT, 2], ref['p'][HAS_CT, 2])[2])
    t_ct = pd.DataFrame(rows).T
    t_ct[L('p QWK tous (Holm)', 'p QWK all (Holm)')] = [fmt_p(v) for v in holm(p_all)]
    t_ct[L('p QWK CT (Holm)', 'p QWK CT (Holm)')] = [fmt_p(v) for v in holm(p_ct)]
    t_ct[L('p DeLong CT (Holm)', 'p DeLong CT (Holm)')] = [fmt_p(v) for v in holm(pd_ct)]
    save_table(t_ct, 'tableS7_ct_encoder' if EN else 'tableauS7_encodeur_ct')

    # Sonde linéaire indépendante de PRISM : moyenne des 16 tokens CT (après ACP) -> T-score / ostéoporose
    from sklearn.linear_model import RidgeCV
    idx_ct = np.where(HAS_CT)[0]
    probe, pr_raw = {}, []
    for bb, feats in ct_feats.items():
        Z = (CTT_REF if bb == CT_REF else ct_tokens(feats)).float().mean(1).numpy()[PAT_CT[idx_ct]]
        t_hat = np.zeros(len(idx_ct)); p_hat = np.zeros(len(idx_ct))
        folds = df.fold.values[idx_ct]
        for k in FOLDS:
            tr, te = folds != k, folds == k
            sc = StandardScaler().fit(Z[tr])
            t_hat[te] = RidgeCV(alphas=np.logspace(-2, 4, 13)).fit(sc.transform(Z[tr]), T[idx_ct][tr]).predict(sc.transform(Z[te]))
            p_hat[te] = LogisticRegression(C=0.1, max_iter=5000).fit(sc.transform(Z[tr]), Y[idx_ct][tr] == 2).predict_proba(sc.transform(Z[te]))[:, 1]
        probe[bb] = dict(t=t_hat, p=p_hat)
    ref_p = probe[CT_REF]
    t_probe = pd.DataFrame({bb: {L('ρ Spearman (T-score)', 'Spearman ρ (T-score)'): spearmanr(v['t'], T[idx_ct]).correlation,
                                 L('EAM T-score', 'T-score MAE'): float(np.mean(np.abs(v['t'] - T[idx_ct]))),
                                 L('AUC ostéoporose', 'Osteoporosis AUC'): roc_auc_score(Y[idx_ct] == 2, v['p']),
                                 L('p DeLong vs principal', 'p DeLong vs main'): (np.nan if bb == CT_REF else
                                     delong_test(Y[idx_ct] == 2, v['p'], ref_p['p'])[2])}
                            for bb, v in probe.items()}).T
    col = L('p DeLong vs principal', 'p DeLong vs main')
    t_probe[col] = [fmt_p(v) for v in holm(t_probe[col].astype(float).values)]
    print(L('Sonde linéaire sur les tokens CT (sous-groupe CT, mêmes plis) :', 'Linear probe on CT tokens (CT subgroup, same folds):'))
    save_table(t_probe, 'tableS8_ct_linear_probe' if EN else 'tableauS8_sonde_ct')

    # Résumé : écart de chaque encodeur au modèle principal
    q_ref_all, q_ref_ct = metrics3(Y, ref['p'])['qwk'], metrics3(Y[HAS_CT], ref['p'][HAS_CT])['qwk']
    ph_all, ph_ct = dict(zip(ct_runs, holm(p_all))), dict(zip(ct_runs, holm(p_ct)))
    for bb in [b for b in ct_runs if b != CT_REF]:
        d_all = metrics3(Y, ct_runs[bb]['p'])['qwk'] - q_ref_all
        d_ct = metrics3(Y[HAS_CT], ct_runs[bb]['p'][HAS_CT])['qwk'] - q_ref_ct
        print(f'{bb:9s} vs {CT_REF} : Δ QWK tous = {d_all:+.3f} (p Holm {ph_all[bb]:.3f}) | Δ QWK CT = {d_ct:+.3f} (p Holm {ph_ct[bb]:.3f})')